# Reproduction of the completed four-class spinal study

This active notebook retains the same 20-run scientific workflow and numerical settings. It embeds the corrected active source files and the author-confirmed study design. Use a **new project directory**: reruns on this already evaluated cohort are reproductions, not new untouched external validations. The original study used a manual, content-independent numeric-range split within each class, fixed before first training, and the author confirms that the external cohort was unused before the original planned final evaluation.

The originally executed source files and pre-correction notebook remain byte-exact under `provenance/`; existing results and checkpoints must not be overwritten. No training or inference was run while preparing this metadata correction. For replay of the original checkpoints use the README CLI and `provenance/original_code` as the frozen source root.


## 1. Środowisko

W Colab wybierz **Środowisko wykonawcze → Zmień typ środowiska → GPU**. Dla batch=100 i 640 px wybierz A100, jeśli jest dostępny. Test techniczny sprawdzi pełny batch każdej architektury przed długimi przebiegami. Nie zmniejsza automatycznie batcha ani rozdzielczości.

Wykonuj komórki kolejno. Procesy obliczeniowe są uruchamiane osobno. PyTorch pochodzi z oficjalnego indeksu CUDA 12.8. Dokładne wersje i GPU zostają zapisane. Przy wznowieniu użyj tego samego środowiska i katalogu projektu.

Instalator przypina także wariant CUDA: `torch==2.11.0+cu128` i `torchvision==0.26.0+cu128`, sprawdzane od razu po instalacji w świeżym procesie. Sam numer publiczny 2.11.0 nie rozróżnia buildów CUDA.


In [ ]:
import sys, subprocess, os
assert sys.version_info >= (3, 11), "Wymagany Python co najmniej 3.11"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
    "torch==2.11.0+cu128", "torchvision==0.26.0+cu128",
    "--index-url", "https://download.pytorch.org/whl/cu128"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
    "torch==2.11.0+cu128", "torchvision==0.26.0+cu128",
    "ultralytics==8.4.163", "numpy==2.2.6", "scipy==1.15.3",
    "pandas==2.2.3", "scikit-learn==1.8.0", "matplotlib==3.10.7",
    "pillow==11.3.0", "opencv-python==5.0.0.93"], check=True)
# Check a fresh process: notebook-kernel imports cannot conceal a different installed build.
subprocess.run([sys.executable, "-c", """
import torch, torchvision, ultralytics
print('torch:', torch.__version__, 'torchvision:', torchvision.__version__,
      'CUDA build:', torch.version.cuda, 'ultralytics:', ultralytics.__version__, flush=True)
assert torch.__version__ == '2.11.0+cu128', f'Oczekiwano torch 2.11.0+cu128; otrzymano {torch.__version__}'
assert torchvision.__version__ == '0.26.0+cu128', f'Oczekiwano torchvision 0.26.0+cu128; otrzymano {torchvision.__version__}'
assert torch.version.cuda == '12.8', f'Oczekiwano CUDA build 12.8; otrzymano {torch.version.cuda}'
assert torch.cuda.is_available(), 'Zainstalowano właściwy build, ale CUDA nie jest dostępna. Sprawdź GPU w Colab.'
print('GPU:', torch.cuda.get_device_name(0), flush=True)
"""], check=True)


## 2. Zapis na Google Drive

Ten nowy katalog jest oddzielony od poprzedniego eksperymentu z innymi transformacjami. Po rozłączeniu wracaj do **tej samej ścieżki poniżej**. Zapis obejmuje wagi, przebiegi, logi, źródłowy ZIP, hashe i wyniki. Nie jest nic publikowane na GitHub ani w Zenodo.


In [ ]:
from google.colab import drive
from pathlib import Path
import json, hashlib, shutil, zipfile, datetime

drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/axspa_spine_reproduction_design_corrected_20261006')
PROJECT.mkdir(parents=True, exist_ok=True)
if (PROJECT / 'runs' / 'training_plan.json').exists():
    prior = json.loads((PROJECT / 'runs' / 'training_plan.json').read_text())
    if prior.get('study_design_scope') != 'future_reproduction':
        raise RuntimeError('Original experiment is immutable. Select a new PROJECT directory.')
CODE = PROJECT / 'code_design_corrected'
CODE.mkdir(exist_ok=True)
DATA = Path('/content/axspa_spine_full_region_v3_data')
RUNS = PROJECT / 'runs'
ANALYSIS = PROJECT / 'analysis'
CAM = PROJECT / 'eigen_cam'
PREFLIGHT = PROJECT / 'preflight'
PREFLIGHT.mkdir(exist_ok=True)

def sha(path):
    with open(path, 'rb') as f:
        return hashlib.file_digest(f, 'sha256').hexdigest()

def run_logged(command, label, cwd=None):
    logs = PROJECT / 'execution_logs'
    logs.mkdir(exist_ok=True)
    stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    path = logs / f'{stamp}_{label}.log'
    with path.open('w', encoding='utf-8') as log:
        log.write(json.dumps({'command':list(map(str,command)), 'cwd':str(cwd) if cwd else None}) + '\n')
        log.flush()
        process = subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
            code = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=20)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
        finally:
            process.stdout.close()
        if code:
            raise subprocess.CalledProcessError(code, command)
    return path

print('Projekt:', PROJECT)


## Install the corrected active sources without changing an existing experiment

The installer checks each embedded SHA256. It refuses a different existing file. The original AMP-repair notebook is retained as provenance; that one-off repair must not be applied to these corrected scripts.


In [ ]:
import base64, zlib
ASSETS = {'eigen_cam.py': {'sha256': '70f66b92dc2145ae8ed8cd45cd11b12bc022c3d2b7477a886c2636b552ed8e32', 'zlib_base64': 'eNq1PWtz28iR3/UrJkhdLeCAsGR791Lc5dZ5td7EVY7tsr1J5bQsFEgOJcQggOAhiVb0O/KD8seuH/MEQEqbu9tKLBKY6Znpmel3N3/7m6d92zxd5eVTWV6Let9dVeXzkyAIXvabvMtWhYxFVXd5VWaFeJVfynJ2/vJPYls1oruSopQ3xV50TZaXciPqJt9lzV789d2bd2dnYldtZJGcnHy6ylsB/8vE21d/EX/vswIgd/m1FBlA3bd5m4jXndhUshVl1YlGrquy7Zp+3Qno2lVNvobRf8ov+0aKrxMAKEWb7aSottt8ncM7HDGpG7nJoQ9M7iZrYDpZ24q2r+siB8h1U62yVQ5jw7eYZi9vs3V3At3g3Vq2LSwhL+u+E50s26qJYX4b0dYwVxhiK7MOxoe5vruWTZHtYUEwnUZewhQlDC26CqBm3QnDyLbwlIbZNtUXWYqbq6qQs/dv/wBw21quuxmM3MrmOi8vAUybf5E8IGAIANfZZgNvAH9vK0Yl4xmexep7KwsAA3sDy5FtN2slTL3nFrBBu6wEXIu26pu1nMHWXEpnsTgqbEotG8DXDqa/go3EnQLYfSET8aEv1SLWhczKVF5nRZ/hcEm9T/CMnMDKdiJNtz1iJk1FvqurBpZewjZSy/bkRD9rLuusaaX+vm6v9cerrL0q8pX+yn/gQbKTXbbJuky/+VtblfpzXWQdTpznAK1kl8OJUG/1d0AM/PulKiW3g73EoXSz9/CVX3T7mhDCz1+WezPxst/Ve9gxUdbctO36zT7dwIZdlrpDXgKeALedg6VYwMccZ5LyCUjXsGtpU1Xdycn5m5cfP6ZvX/7p1UexEBdBCUvJiiAWQQWnqdrm3R6/AMqydl/CYDvzzPu+PPnx9cf3b17+1YH11sB6h7Dqqz2cDvz63oFmn370nyxPTk42civaq+zZ19+k27yQIaJtTtiKxOx7wEAzPxHw3wYIQtvBoGoLE+4URvT2Ju+uCONJVcsyDJpVECEir+CUF5Ih4H9ITNZXffkZ0ChyQGRYZLvVJpurlkkjs014dvrshXgi8E8Ui1UQRBaCnUvS14jxkODxNGBX+qbU76/kLX+CSfJCs6bJYD/5IX2Zw14n5YY++wuGQ/9HWCoip4bDtYFjI5lMnM+qZgN3ZYVIhDunaAhhRjKp6EtYFhCxqm8TvD0IkMYABMKAWQtkr+P39JgnEz0C0/7a8ZokGzi2bXgX0EyDuShyvbqEHgEOA5o+vOPH9C2Bpd7HHmJH/8GyuvSz3LeLT00vo0SWeLDDaGouDLqrCC0hoWgRnAfR47bmpoHjkOJ6nCNI96qXc7yltDtv4Xrz9tBh406dvO1cTFCfWMCmIKkCXOf54qesaOFZDue/7BbPHlg2bFZRVDdpmZXcMxK/E8EvZYBQAQNAPxZB321nvw/0/CUyzHSd7ZAC7FLgNvk108XQ+Tw6bl1fF/LCPoyF+xl53NIcx/MCuNwsuyyBbuRrsc0bOCbAiMt1XmfFbF0BgSphecj//sbcAtimOP/XP//4r3/+xXK1EwJ4LpGQCZmtr/QruJpAz4HbVMD2DDesqzanySfi459/FEwLWxj+ljiJ2t2/93mDZBW5YAEcAJlUtmqrou/gQZVtiOaWzCVp5k1+edUJZE49dBDXMOOqIWjAXVdSjQuiQ8jt9eRu8hIuWcksHYg+HEvxVl6ymGGXTheRwK2LvK6ZaX+RTUV3mCSaCuCN+uHaWhBCuMPFaSzOlolAwYaA0fLh9l4D+lTrjQRE7oBd07YgdCPf9C2CJlmFpgRUoshWsoBNQGA/Sjg0soHb85TkoAw2T2+T2Oey2OBC4SzOaOK7rIYDeAtCzjrvQBhD6Qmu2SbRJ4TpjKYxirLY06eo2ALebmFLum9e8N3MgTLCoct34jcL8ZzkibwMM0U9xHfiDJ+p7xdnS6DN+suzJbx+ZskziC2w5D/jDXzVNFUTboNXtygBSSte6QPpzewSsHWnoN4HZl6IRFxMUYTwJ2+3gGakNVF0ZExgi+WMW7qDfAvQBIq0N8CZNOI3aqx1LIDc3ADu1CTo6S7rmvyWHsKW4OMQGwICbqLkU7Ku6r2iy6rlbKE+gUSTAQ5v83YBR+izlDXgV5HR4eLKfcidnEXRhtNG4qc2DHF20XAHnz+LLBqYwmLzmP5V04vFnUftgo05dcAQcELAH4DvAgGF7wGgWl+1Vm8ZnDV9PgOfdAYgL+LVTvU9Ton6tgDpYglw6eqm9RoeN3lWrlFAyugyQIvT5HQADW9XCs2uqiZVk4B2SPVxjtlNau9pCmd0EsawWXY7bHZPn9LYEB/gM1eM7CIHPeUyaa83aktise2LIqUvIEsrdsBHxhDcBfS/OF2654EXAW9AXsSjCyQRZkKfVm1oukaROQ3m2QX3xZt1ag+EHe3JQszOmAlaqrUQasLiv2zTyJxaOj3cR5NVWi/MKd/1u9BCihFX+kzTS2hJhy3UXbGXlgGA48F7A/Sp6QQr0h+/F6cCTpM0pzkt8s/SgFPChCwr2NKsI6zxgNC+hckZHvHE7JgaHdUGaG0PuH+4zQIW/gE4cmr1k4v510sQZkiWipyeRw80z9qAOAVK+UQ8A6Q4i4sQM+5iCTP+9KavAT9wmk1eCLVV5nGCtDyKjnSjCzLuRns8Wnl3Uw2xlpYya4p9Cqx4A5BWVVWE3pUsQCUwGwen4YyYJNHzdVG10sWYvZTAaGLRwBYszuTsP0H0Vh/PHJqn5nfviphwIoH5IZkMHTIZO5dl8jUeJSXLAZphVikIGDuQztqwqW5aFqsvSCKL8WClEnkNPM+JALwgcc62sVJb1QCf7uZK6mnJ0kLiKyrkMOmtbECoBOELxbtEvFJg6SvKKSX0LAoltb2tkBJv8w12ma0yNGTAdIE0faskC1GVQK5hGXkJMHnwYv8UpET9WQlFAD7f5siXO7guWSJedmIHoiUAMB2FwgFN1ZsgL0fRwWstSGbXWV7QAgfriu1K0PqgsNDIQolfKOxbTRL49KZHMQeusSBrBoyLtpGcjRl6Z0jX2marhloO5CDFYE/Fdwtnv/Dbi2Oyg9N01wM2QBJdye5GwuxP6dy+UBIDaTiAvgVpSXJDxwRZ/X7BSq1o5qK5CPQiU1xksDT0Hm/F3fg96cgNispqgPsIxTJsrR4cFX1+NHjzkNsiQGuwEDhXtQ42LcmNe8LRuLA80Ro7HSwE0GTlpQxxKo5RwxXF1oAgsoO0CKEZrQXXDUvuQOhICSqsF4gzD4DIhZcoL49eLs0QyCnNKL5dQK8kyUDeLzfh3ZMntinRlsDY0dLLpuprFHjUUQ/uGR1q7x89fdibwaR5rnArW9AILbj/U3wqsEfQeQBjhGUc0ZsfCvkOTfOw6jXUqPXwyojrW7oMR4+0B+veIAROo13U3J34xEy/Xxyc6grE2M9udwA8ng1SBRgN5/uIlQKIyGuF/ZJsQ28O3G5zEOVtpw5iM334DFkO7g9gyTOfaMDacJdtZUr0kYYP0YLayo4Mj9qC4k1wjsSeOBW+nCvQBWwbfg+9toZQwVPg1KnW5sMItcEgSQI6wvCyzpquPaoD/lziVC1JojkLntGdN6rW/UBwrYprOlHeskCYgvYk2OL70FOmdCecroHaVR4A2zM6OuXXNEXZrkF8boWCIBiz01OemASZVEfj/ARP31bdT1Vfbng43cfbbP1QbTbbjfMvRMDTDYgredGGwLvLFq3jrbWSHTcqEZWHY7C0Mso7FBnyEtg4WvThEzkt2CXkWzdAYF1fXectmT5Kw+Kvsrw0hk42ndcgf6R2dtp8/hGffyAXyPtsE4tPH16+/fjTuw9/St9/ePfp3fm7NxaGNxh3dwCCEv/pRJFaGB4OyiXgpOsaBydw6eyXICZN0ipc1C1vHaviNE89Jy+HRos26KBfzMD+FgWenoQTQGBfOr6iin1I6pCU2Y6JOUmgoKGlKT5KU3a08fWHeTH5lsp2kmJr6jbCX/Kp+qR8WJ+St+qMyKVepB5nCB75lg/9cUYcx8s1moqZiJnG8IwoOw8hwbk2NEdqgFo0AoxwXxjpnSCngfMS5z4+NxcBvgP6PtX8mxenx3bYujvZkilbd6HQeVbnt2h71I495cWbARuc1dnGnoRAK6ZZ6dvheErPlmQYOmTHabvNgV7w5lAn1LQBJputcLHh85hoNPQZP1QGJxwJROHTo8Kks+uNlDNtgfUokfL9aAMAGok26JLdyOuc7W4KJea0+d2nt3JfFZXfMDBHOtR2M/LmSPQyh4xUB3HTo10E2DBYTqDSZ/KIqPEotAmPGATaTY9xFNtvqmwDuEYH9wDFm3wL16hVVNEeTHPqULntqnVVBB4HYazwrPHKKU7C/ipgjGlzuQrJh52ur27mHpvAvv4TgDPyX9iv8ynjs4F95OwOTM/asnxKJOr546jShz/8IM7/+BftPFdkaMqmDCtGkSJDW1K3uZjHyrqI/y7F72jZg6dRQniu0VJxFotnsTh9yD4No0R02+ADG2BAwJ6dydkL8xDNK2gLSU5PT8+OHYvXyJdbOdNHwoQuAIWs+q7NNxLdFGdLhwd1FdlwL7Wkhe7GJms7/3jAjNE3grOFNcXiTDuzQOYhN791XYR4IUnCiFWUQeq66Tr0+HT8/pBrbZWtP4MwnNIeKVBqgPTzDQAACQa/PEKEsZ+tDIMxDHg1TKCIZdMV8J9YLYp9VFmn3Ec12SaQhAEqxWWTbazDVoscKIUoQYPxsklpA2L73ThtFmT2Bl1P2TS3vFfpVVV9DjncQi2dF0zrHLg2yQXGLuUFNYO74Co1xBZhm5HUroG3q9NO80yYBQ8c5nyoPsEFVGfq51KTLvSCzdRofHtowkHk6KDuorVaxF0SFEPXV2HEtxr+ruse/qVICvxGBvHIYkMh6jg+YjzXMNaU33caB9x+gAO8avzCUJgXD+Dlo9Kv1IEGTXovGziVGyXZ07XJxA9Ia7Q7izExhTDjcVU4U5P5NThbSSDvaKv3bk+ixctUBUDhTWWc2uOm/BAU1rNQCxp3pE7urnC3rtlbVPljoxUptKsl9s94N8IetQyHh1C2fdHhFUFaokO4QqYmC1BIQoewRLF48sSnDjzkFh00hTs5whAsbFcZddAs3H+s7QnegSbrGjk4vXd67/zXag38cEi0gQJhCNJQdiJCA/oVmlWJGj91o9ZcxUGHjJCTEjj6tSzxdKujha4oS2b8RZBxazR3TTdg3bceV6X1mNM5eoMMa/D22XwZTXl6D6w5Iyt+IRWnQpsTCNg96AuAiGGYXeBYGgi9KLhjDF97SDnzR/0Tka3BJdX27TVLUF5IoBrQewYoHY6eoNL/wG112ekg7PCWNsXdC2aucEM2eAEpxjHUpmzfv8CUwrPjbOqcfA3Wz4A6vGV/BBWAA2L5lO2yWpDWAqpyR6wRdRvZ4v6QaaFHzJBTfJt9luLH96/JQLOSBQYjDrjgLuvqosJYPe3uVl+THsSi4OXlpd7GYfuk3uMn1NfromOyRvYv4pYXkjRTSfY/bdaHoyAvRuYysm1+lvvlQMLA/vAYIYTGmhu7trVoORiVPlFH/gRd1UsYmj4pIy5gaVdiF5TXkALQy2iqq6ZPFLq6wLUmao/hDyqLi1CDeyJeJCBvGXhAUJ6IZ8l/noIAepr8/mugfsB54BAsAo6PpaBb7XfomaIzcLREAn7yDdomQgdirCfPvQrZdZIM0RR2F2SrNRzGyysFVFlhUwyQAkJjlgaXdkfeVA12EMtHQ+he2lk07jdgBTmokbgEWsqFM64LD3TefmVWxoL3DdAOYBanyenzaOBi3HY+TqAzHruQxsIwHm2nNqSE4o+O9zlbjodJ8l17Vd3om3sRMMsFIRp1Pg4PrQoiO4sAXaOydcUpM/QIjjYUPRYKbgAGmNDBx5nFDHiAbGKG0CxBCyh+SEFEDofr6vhQlRhLp86K38RMs85KdjSoXr4LpgNyh6dM+z0Dusy67+SVNneWXeGg8OgHY9zjGrq8K+BSBeEdT+A+Enc87v3c+h1/Ke+8eNkLPYmvrLvkq+XyPjgWChj8Ur5cdxha7crGQNVBn6Ir/Xu4qdV6EeDkJjfZnbCxMs21ciI3h6dpXUoPTlNsYaI1Ol+c3g4z+mp5AOw8eb6999bjUjG4gDXPHTl7UTUYr7C3uQG/lG/lzWQqQMABLYp4oliBWzu5hGnQc8w3mAbtzPbsuaZeFR9fYES8XUNDHRBWDrafqWB73srkl3KI2GGY5fvzlxh4B4JaXhQ9+crRzqV5VfwL2YbXWd/CsBicl5Vs9kqGkB+BkuCHCqRp9ngLpA7EtykB4jE5Bf9vixQH1zg8L7d0tELekfHBqjFhod5smfhrGUc8hRPsRNAStPQORJ57+H8CnYJ40H7UPAGogTeZ7FrCx5CGBDAL+D9MCPgH7ELVLIKbK2CFgwXoPpvtw32Qw3P8Cvf2hMELGDZBoxstlz5pAVAHKKVmCeGTWEz57m4k0pBWfzXmRn6ecmA4+fQOWl3GsSsTsmW+u2y/kHR5EA6ac9eSxkLqfhpYiRS+f3N6erCnjTpIrU7Dph7xDxLvAQCZ2w5B0MkwadOXPGW/I4nECM8G3YAcjMGf6DDdztD6xWlLXkYSpj2xSTUrMO9gr6eKMTWwFzpS+gNcQzagYRiP5/qL+a6m+SZWWRSxsOyF7Vvq84TKEWZwP/pmBkxxh4F9uEvIzN2MEYo8iDAAGa4jaD0qptkJ5VhXVyRwq2AVVCoxqKeyOUPo/scLvJJu8NA2L8hB5UfNED7ev36j5Xhyg/5bigBBsq91f7pISGvGNjbTC9SwJiv2QJmM3xBZwInWE+HkaS9++ByOHp6/o6b1n1tJkdTMHOAeIUirBGGYPMBB3RcgIXirkzrXJZEgPXVtGJFCi14U9yXK1Zu8mfAro7/3FXXVhut31A/ObENR73vt7ZK7utt/S2HjGXFA02Iu7pzBBi5nRSd+jcdZdTFg3Bwg/c7zEfo05wiujQ1NgRGbiiYJR2F9pfRQjls3F/upc5wx62XgUEcSdmRAvBoY8grSILIs2AbMYBzfEMAx4ddXeSiIwR+BXYAUzDKI70CtUl14ys8iZATkZTYhOJjLZaOB6Jtre7j34lP0UDOcQ4KZNuGk7dZze7w3ZnlaQJGtP7cgAqsotCmQ0b0jnOq4GC9sSF0njkh6wb4LbORFYw0aPTBR9FxgWhwTOY5ndHaFgdppDc0xjhtJzcRF4/JQMoNaodfa8YK+UF7Qab+N1yt69AKdVVkIe5XS4mNeuV/9xX4nTl3frP/ye3HmTlnHzA4W2O8w0v/Mhsk+vD2vpmaNI2pW4ibIULpqjzEZaMj01+SHuA9wSAFyw5P00NQ+ZteUZKyUJBUMS6ouGkTQhA38kkzPeddOWvc4fQ0NyhiheSRQKhaHA7kmiCPDtMtSNGD54K39yJNhhxxSOQxnJaKIAUUA6itvCl8t7wcBmmj1nYhM9milCeY8mh3jBNDqgN0gMtzVFbc0Y/KtsOyHxmbsEzBcGhO90LWQGldALH5+8+nDyzd//fT6/GP651cfPr5+99Y7PKMc4AT9nehzDBw5ICCMT8CawrtnH94S96e8tGr9GdD4syNd3E1AvAdB6zpvqhKFMuc0odsixrQR5Q9BC8hovaGLvViJzQv+450pDQRW5a3zwVtrQlVga0icQ91jw2HfOll/oDivr+T6c13lZjmoes69hcFisHGIfhjN/9Gv235eBMqWPgxhIjcOxxWQjYmEq9QEONGHi5yjP3M/3JXeRdGSDoB149HjWHmByRhE+Rfc2sQLuOMA9hxp+WiwwLlBgiInnMXrXEIXsr58vxWvbMKdFWEoT96h+1+15EjN+ks8NLAdP71//sy6oFnGZj8W4IYcX6R0LZB20qdocFhioBHFVieuKsDq68mUdx2mtjjDKA3QZXa6Hyq0+jNcqxVQDzeF6bfiNfC9nMIKnm57dU+UmGy9/+awKnek9iKiiQF9hiqd8aBf7yj11STr4nQ5psLkB3T9f54v0on5c8cGfNNrkL20ycgL/PNhPMrHNAgBRCzlBnMb8X7/CTWYCZSpY7RC2tM68/XmcHCi3EvNUDsh+Sm54/ARBc/xs4vZ2dKJKETScq7v7iM9d0PK4bvRGGMwb5XNe+5TBuU1Xwg9nWdL57lPGogssIGE4wAQnu9pxrf0uUdeR4lTummrYC5dx647zGNctOdVX7DTsC/zv/fom0TduIMF0Q6r5Wh/JS3KWykNZZzrPLB2uqpnuDvoZAjuOBCTnuIe8UrS9D4ZvkEjBe+gsvo6YVwm4ssgczoseHAdbKCqdn0i3XEs0l7AA5BCZ0X3IrxzFnMfBS6MQJVIIZpQlXNNPB42wKqgkxYza1aMcL4Exs1lVOvdZ1Ssa+hddpxDi8YwIEFp9dlJqSXPRJtC25F5kb0WbTBopyDr4MusYTFL1xRJyuom1GVFkr5bRyB+V4hH9AQrc19JqSGx0mqdECCtZBKfiU2Sg/GMacIX88iLM0di/z+RXIcO6djNhrYxBMMwLx3hpcO6YjEgVi49diM7/3dKGyZ1cKAjxQs7OaBO4qyv9Qzy+L7BJDwzmygaKSmD3qyvHFBRfC1xQuey9lhH6bIZi9P610AyJTHOhEh7YZ5KWHVUSzxL04qCCB6qZ7ENQlir+AfsX5eJ+h+LOx/d8+Sby/voW0oCaLsclD4M/a9URMVQVecgSj+M9Da2dMqNhUJKhXMFzHh5p7Go11mqEngfVz3DzXVvie96B45MlQkCULHTZmxKF4EOfloqxj2amgaxsN9OlyB2MLQPskVNCQMifnj95vXbVy8/HE3C/634b9lUtpoDbHALs91hpkFfku1TbijmAm58riIV7RKp+MRVdVMmBuKVzMi5jPdVm04vAri3qypYhgoR0UWSJLGYP19O5LYaUMrBjKEusFaOMyNbidqICzd7eskCeHiafHMqntCWY3DCC/yi5+Ro2p2kPHGms+kd0bz56bPNfcpHVpvNvlpezM+euf7M4bbBzImz4IhPxLOvv46cNfXw5vdRRI6a0KH36EC6w0ncqzBF9BlFD46hMfLvjKP6DkbClHXo8yVFi2RDlcYO9KeZtElZfwmOl6MhqsvLWsAtU+KIe0kWbimNo7DoDq6da7gY3Mqjve2NWrgXGz+rg7hQf4/D8ejowjcY+YGFvHFUVEobXjLU3PJLjAMcpH9y+CBJGwvWHHVDzilxyAeyaq9wAOfdm/AEOr/BXPPuidQ9Nv4M4xmGtSg81jyf5NeuTVm1MN+H8PQS2eY09y1QsX0Pq01v8k13lV6RLo+FDeyrAVDHDjwfW4a1cZs0AWjgqNsXo8bD+ToMdX6MA2ub+dQgR7qNh3OljfnACGyqOYjA90l6fTRTnyj+QGOomhJOv73DR5l/WfBchMJpMIDmeVWc0X13y3DDWBonWR36OLJ67L9MVdUvR3wfrsbSlpSYHyZr6Bpit7Z+2KAdT9Yrp3Y7RJSlSWPIDr2yY/gdJoZwRYHhzq8z3G/FyAYvedpIp+eaQ7VPR9wCC/IxST/Q1CX4EzVnmJ6nSM/HfSeJ/f2AINmcY/rq0itWMpzEdrh6sXBCz+aCsjhMENlcc/v4gWtxrwMgChUE60anWu3GUafI8+/40GiyXvTfF0wq4SXFZvKuU5LeuaFkw+gytlJaMyz54rhmKS5AVYRM+EmqDcZER9Q7r5n6EEY257wGnQYt8RQ16hpg8Q6hMcd84AxU/EpBv1ReMcf6cPjJOrjdKoXOzC/UQLjKI7Zu1UorlWRN9wrbcIQbFyxgXycOr8uq4nMMxpJTwVSm+Cr6uY/VcXXOZrCmEIpNClovcg/WjWM7un7zOF3ZLW6zbvLaoSKuc4Wy0FP6nKZe9Rllk6ZXSOmgo/Z6K8KnWzyalrJdmw0pPs9BDuG4AObufsZurSE01BJ2mpAiJ337FZb03GH6NHpgE+UDcIoxBkpCd9FudfEU1PwOFA+cG2vcTjOPytfAqdZ45YOfUNgRXVXPCnmNZWNd0xUcvx0Ah/2CA6AsuWh10UY8FOtBN1G5haoBHP7e5mAemMG/z4QCr0ottLqbEnhwae90XlfhpEmee+bJ99reFXPInFcdGMTGlryKKrnCoYgjeUgby1KcHJJW+HPEpOZmmZvjNHgyGMO32lFS6pyzeB0pxW+ESaVzyuk9IJlo7rWugLhicSmaR+AGsM7coscqN8ha6n5FrWK2FDRY08vLVDVeqE3e1siHhkIB9OYS0ynXLE6VXLpuqhrr8oLkncryCt1BeNuAP6dYqliZS2E9Ex6QQHlHGE9ZZ71x4/ZOxVMnhBBu6VW1GR8/phA6TNKWy8NEwRyNkSlWAzkwK5ZJAmWaouKZQH/VlZzpspriX//USSImwVqXnltTjU4k07IZFukMxrKv0uFw0HPd07X4qSbkLKQK1kdLcQZTtfls5Usc5Y2q60mq2iUxDIBmyn6CFlPmXb8he/0Obq1JE/9W+DU9qXBLa0t6BseuC62vyGsxVb1TFflEJ+I1ZsaubL6Rkty/NbUMKeRVW2ew10isM9YfHPMHoBcYie9H5+OIPG/vbm2AE5Ytpb8P8bgDGepKwXxbiT9kPdAwtJy5x5jKetMBnZkO3/L1Wul5qDsmnGlOkwQciSxGbELCdQMozhqOxYvT/yCLDj79nfgGvuXjlGMv/n1QAjLf5aoGOA702o0g1ocWtgDjsqgGp/K/zpDaIO2eCiymsHpLVlR5bY0X2G+4Sdllhp5jNJg13RpQXwBSSh8L7nU3yvrEPacIMVKOUUXBTD+MrIhNbgIc/J4kAQy0mSi5NKxbZWr5MJyhPvIYqL/5lVAbpo/DgnOFvM3XFRCs+gpQ7dkejhegU/WFoclDVd34BiEh8AejwmOxqcKIYWCxX9uNqjhPFrUzm5WM7o9ymxjkcWImaRukRq5aZBxc1U7dMQ66sZXigOlJTOTVZe50QACXoOKKeFjDshUM7cCJmgxtRnFx6jmCv7uPjStqLoyKFKiId1LCUHvEv25pQ6c69sEweDtSgg2DWKkRrFIM+6kR0QWESO5uuyBy62mfHM/P8OVCpzLFMfWimK2L1vFVBDYuRYQodTBBoPxEfc5aCV9Po0Sc6/KFbulCrxAgDh8M6lWqnQcuMH0PuBAj8i90zR0tYZi4wF/bcoqT81F8VZIsBct7qDqiA3pwpbh6X18jn8EbZMYgL8Lhe0NVjqlWmQscY2XrjOJD+RgaJuIEfjDO2KGrf1VD/eaF8tEq0RFj7xzgWd9VILoAen/dD2BYl7CWmT2E6OoM2IGzza8MY5kRK1OGGbfTj7wsOoNubCMC0UzJlY1uYD9EtVJ5AqpQuadFOMBrOg6IS5Ig9OF7TEV20sG5SHrbOZqaA32QH62UN0dhGwRloNpGs9nTTNZ+ao8D2M1+sHk+xG0P5PfYH7bAooFUiMvly/7BRfJ5zYcRgKyLnvZ0xJ0ToSMhOdYiB1EjVj+eELs/dOKAJkUyHugadICqbXeDq1aBD+qHWlR8udrGbI07kZV7yrLHtA9MiYoda82wpr+f+6utUhN+gFg8ECHuGexHXhHXtO7bwGPx5GIb1CnVyroPTLTMID9kOQbpm5QHyngwbcudtL6OQVtD6sBYGo9MoEtbzfUIyyISlKzba2A+/MshaFRDHgLXcBFM/d7C1M+KEN/CoA+AlPwI+vlf6EHI7WK4mLLYkBq+8JKibV9mfXiTZOPkxlr7Jtc6JH7te522sPtoqfss95qjX2CSupuWro9SvtVPuOH9CJL+MY07Z+/n1l7qR1OY4MrYHA0bXeJWFL33M1PdJYMkGeLAbtANnvk7tGmReOYWUnR2Mormox+OiYYnhg2tGKVd6iq5blxPc1lUqzB4EkTsh8bhTA7LQ5IPOdDh1KZMOrTQo5bgxzlxPPmdKyneK1HF/BCAI9wYNg48byrtxhRoRPlKZfVRKDAHbaR6au1kRU4bIWwfHnGXOsKk7eAUSVLlIeDUx16en1+G+uV6LeuOFUhjwnWjSxRb4p/10HKbYvZIaEkOM1UhNAjWIxbDSdL+KKNzqtvyDnndoSf9cAsij6viG6D0U0Asi45ogI7Xwl9natcVRdhN/S5T6KEaJ2UeII/TR8Yb1wRfmukkQD1DbX+nYCFrgh8WjfP7KDSmRtxLVYQbAUGDohKKfzUc3A4Gczzx7fzAVk8J5rzFqPToHCJ93Hss0zfCpTt//I7909PAFObHs6/PB4J4ijpqOXUcdGv/OHgwHj4OuGXGrXJky6jCjdfusZsygG63YPzSdS1Egyj142BN3PAoM+DoRr/3bqzeK4vZURLeAX0Ldsjkb0r55dgFdx5xW2dH+YG/nw7AR+2mAkpY8e6tFlAIRf5NmW53tMalE2BHAz7VhGumQFOCFIDTP/DFzvxgMnXyMQTHm/bF9JSXRxMe9e7SrLjQ3wbNtyh429ADk4aL7hy/fDivFENRiBkGS1VJnG8CV+XwrreqzU5nnl7b0lKU0MGj2GJZ/J1C+q2SbxaHzfTJnGwwvCIOOJeEeADdbfBIR/Soi+PYIiz9pZh3fYUeQLSzDGNdUHW1p9bq/BLARM/DBbenGj9UgFvX37Z9Z54BxJ5lFJfsbK1zDoTNgNJUYNqPqKerkUosWtEkyirs/VQsW1nX9wNq6c34Qg9ToQNciO6exJ9qsT+/RYqGZjgNkFQNdjBQwpHmBoRq7TMsfDLiVwboowichsscY3QLogevyfHaveNEaD1txQxWsqi4WCgZO3Sa4ShbjCv8DnBE6tqv1tRQS0TjtQ7Q0wrbB9a8uLn+yUBu50TSQ2/OI2MgbkCJrWGfPZDMyUqAqlofmbqz5pGqhEdbH/hvzkwK9GTS2UNFlAHg0+9IMvj+6XdkJfgeg4ncrCGy9jnhpNPFJNTPStFSXZ9BFNvnJnnWvHSRYHsNhJOEam+F7pofTCT+4Fsz/Th2WudsWxUb/RMyfvLw4FcuceAM7lDV5reh+cFJc3XcX44kD0nmhUs+mLBNlk1reLK/1MeCERo0FcOcIpD6VJo4rHGApzf9QWhndiS00/FvmODOzA/u9OI03YPhh1kOa46MRxjGg11wpgWO55gdltFBo9N9ZH6Dk3+Y5MhPfiC+/GL2aF2ok0O/zxA5+BvN/8CE2M5AoVyD34pQt/gp9YoSZW/gmO17N3eM1sLyy9enOtMOAKkX5jd4VDtKXOqcXnpxjy1UEY9yzth/oIi2KlkBmhpORxWEGmtmZf3lEA9Dop22/RYU+jCgwEOHmOMv3QHPCnX/WP3gaZ2vPxc6QxTpN1v0vNK7/EOv7YWn49jfSjum7ChpTgOwd8HrjcfJvpo6SQfg6bsygmbExMfD2qd4wyZgeT7ah+FNVJHXQ4xyo5xclotm9H4wWBT9mgIP5x//TDL82/f/jR6f7BIUAFsGVXsoKchS2RpYFZj7AhaZuz0bxFxMVn4aW3+mA/5808skpJGaOQ3J0S6n4QylwRRkj2lQg6v0SHB4lybB6Ut2BI4WHSf7u3LlNIwpnWh+RCGansiUHjp/UFGdgvXkiWObmxwMnjdajuewBcfthGF36Fkg97m8zqu+LfapKc6VMnGczJoJ4L4RuI9ozKLfuqIS3vRDcWTahPuTba7xuJMDdYU/VKzJ7oxMYDZOxPtdQarnoPFIqZdt7FweZQZGty3c8MvruS3SO1GtzGacB4GpT0E5/LNtBhR62xeCsYBVv0CTInkcxTvtV2MxZYbikbjss8YpXE9yl/WyKDimfhdioFXQU/yqY471d5o//hYAUHuTTI4mgZRep6lKJqe20cn/AMDUes0='}, 'eigen_cam_replay.py': {'sha256': 'd0c0e653b945a619901018008ed2f8367302481bebf8e0f949984e73f5038a3a', 'zlib_base64': 'eNq9fe2S28ax6P99CgSpWyZlEPsh2SdFha4rr2VHdRVJJclO5dAsFEiCuzgCAQQAd5fau8+RB8qL3f6aLwAEV45zVYmXAGZ6emZ6unt6unv++IfTXV2dLtP8NMlvvHLfXBf50xPf91/s1mkTL7Mk8IqySYs8zryX6VWSTy5f/NXbFJXXXCdentxme6+p4jRP1l5Zpdu42nt/f/v67fm5ty3WSRaenHy8TmsP/hd7b17+zfvHLs4AcpPeJF4MUPd1Wofeq8ZbF0nt5UXjVcmqyOum2q0aD6o2RZWuoPUf06tdlXjfhAAw8ep4m3jFZpOuUviGLYZllaxTqAPI3cYVoBPXtVfvyjJLAXJZFct4mULb8BQQ9sldvGpOoBp8WyV1DV1I83LXeE2S10UVAH5rry4BV2hik8QNtA+4vr1JqizeQ4cAnSq5AhQTaNprCoAaNycMI97AW2pmUxWfk9y7vS6yZPLuzU8Aty6TVTOBluukuknzKwBTp58TbhBGCACX8XoNX2D83hQ8lDzO8C6Q5zrJAAzMDXQnqZtJnQDqOy4BE7SNcxhrry521SqZwNRcJVZnsVWYlDKpYLy2gP4SJhJnCmDvsiT0aNoQCZoqqBenFf6hrkO/TqRfyU2c7WKYpa9qb1VsyyxpYGaqNY3JMm5W10n93EudCQYgK5zDGygi9aEbIRIegt16UbTZ4XBHkZduy6KC8cyhKhWrT07Uu+qqjKs6Uc+r+kb9vI7r6yxdqkf+Ay/CbdLE67iJ1Zf/qYtc/Qa0GhwN9VzU6le9rxkxqJo0KdCefFDPMAXw389FnnA5oBpsXxV7B4/8odmXNPT8/kW+173Jd9tyD7Th5SUXrZvdeh+tgTSuclUhzYGsYBabyIxb4MHPFDGJeE6iFdBHVBVFc3Jy+frFhw/Rmxd/ffnBm3lzP4f+xZkfeH4BdFts0maPDzCOcb3PobGtfuc8L05+ePXh3esXf7dgvdGw3iKs8noPdIiP7yxo5u0H983i5ORknWy8+jq++ObbaJNmyQiHbUqjNfYm38EIVNMTD/6tgfXUDTQq8xpypdGYvt6mzTWNeFiUST7yq6U/xoG8hvWUJQwB/yHbWl3v8k8wjECRSTXK4u1yHU+lZFgl8Xp0fnbxzHvi4Z9x4C19f2wgGFzCXYkjPiJ4jAbMyq7K1ffr5I5/AZLc0biqYphPfkkPU5jrMF/Tb7fDsBL+Al3FwSmBuNZANgkzpMsJLS5YrzCIsLqFW9HIJMyUdjl0C9hlsatDXFIIkNqAAYQG4xoYbMPf6TUjM37ESLt9x7UTroFs69G9T5j6Uy9LVe9CegVj6BP68I1f01MIXX0InIHt/INuNdGnZF/PPla7ZBwmORL2aNyHC4NuChqWEQ3RzL/0x4+bmtsKyCHC/lgkSOtql0xxldLsvIHlzdNDxMaVmuSusUeC6gQeTAryLxjrNJ39GGc1vEuB/vNmdnGk2zBZWVbcRnmcc82x97Xn/5r7CBVGAPjHzN81m8mffIV/gqI5WsVb5ADbCORaesPMcmT97pBbswN2PTcvA8/+jdJ0ocnxMgN5OomvcuAb6crbpBWQCYj8fJWWcTZB1g/Dk+O74n9YLoGA9i7/9c+//OuffzPy84QAXibIyLwkXl2rT7A0gcmDXCtAwGq5WxZ1SsiH3odffvCYF9bQ/B3JLJndf+zSCtkqytsMxAKKw3hZF9kOhFFWxGviuTnLY8K8Sq+uga/D+x1U8G4A46IiaCDHl4m0C5JvxOUVcrdpDossZ+UBmD6QpfcmuWKFxnSdFiKBW2VpWbJ68DmpClrDpDsVAK9TD/tWg7rDFeZngXe+YFlMwKj7sHpvYPik9BrkbbUFxYCmBaFrQburETRpRYQScIksXiYZTAIC+yEBokkqWD2npHHFMHlqmrx9mmRr7CjQ4oQQ38YlEOAdqFOrtAG1D/U0WGbrUFEI8xnFY4SzGOoTLjaDrxuYkubbZ7w2U+CMQHTp1vvDzHtKmkuaj2LhHt6fvXN8J8/z8wXwZvVwsYDPF4Y9g4IEXf4FV+DLqiqq0cZ/eYe6VmIUOUWQDmZXMFr3AvXB13jhIGJnsmwEf9J6A8OMvGY8HmgTxGI+4ZJ2I88BmofK8y1IJjXwa2lrFXjAbm5h7AQJeruNmyq9o5cwJfh6hAVhAG7H4cdwVZR74ctScjKTX6DmxDCGd2k9AxL6lCQljK+w0Xbn8v2IK1mdogmnicRf9WiE2I3bM/j0YmyGgTksFg/ov4Je4N073M5fa6oDgYAIgXwAuQsMFJ59GGq11Go1ZUBrij59l3X6WUJLO1LrOCLuWwOk+QLg0tKNyhW8rtI4X6GCFNNigBJn4VkLGq6uCIpdF1UkSEA55PqIY3wbmXUaAY32wmgXi+/axR7oVxRo5gNy5poHO0thR3QV1jdrmZLA2+yyLKIH0NpFHDDJaIY7g/rzs4VND9wJ+AL6IpIusETAhH4t65GuOh5ratDv5lwXV9aZIQjT2pOZNzlnIWi41swThL3/bYqONdUS9XAdxVapv4BTut1tRwZSgGOlaJo+QkkitpGqirWUDgASD75roKe6EvRI/fzOO/OAmhJNzVGWfko0OFEmkryAKcV9jG4QyteAnJYRT/SMSeu4l4DShsBd4tYdmLkEMEC16s18+s0ClBnSpcZWzUGCZqw1iDPglE+8CxgUq3NjHBm7szQyLnr9y4BfWMV6F4RMlX4dIi8fjweq0QLpVqM57vS8uS3aoxblSVxl+whE8RogLYsiGzlLMoMtgZ44oIZzEpLEz1dZUSf2iJlFCYIm8CqYgtl5MvkvUL3l57nF8wS/B1vFBIoE4YdscmSxycBaLL2fkZREl4NhBqwiUDBwM12PquK2ZrV6ThpZgIQVJShr4H1KDOAZqXOmjNHaigrkdDMVracmmw6pr7j1B6Q3sE8HOgI1BdS70HspYOkR9ZQcamaZaG1vCuTEm3SNVSbLGE0mgC6wpueiWXhFDuwaupHmAJMbz/anoCWq36IUAfh0k6JcbmC5xKH3ovG2oFoCAF3RkzEgVB0EuTvCB2+UIhnfxGlGHWz1KzA9QTuHjEKVZKJ+obJvdpIgp9c7VHNgGXtkN4F20QqTstlEzQzttTbxsqKSLT1IBOyZ9+eZNV/49GxId7CKbncwGqCJLpPmNgHsz4hun4nGoCwsM9olJWsiExT1+xlvar1q6lVzX3Uywk76C83vcVXcd7/THrlCVVkaeBijWoal5cWg6vODHjdncGsEaAwWHuIq/WAjVrK2KRyNC4sTtWMnwkIAVZxfJSNExTJq2KrYCgaI7CA1Qqg6fcF+Q5cbUDoiggr9BebMDeDgwkfUlzsfF7oJlJS6FdcuoHoSxqDv5+vR/ZMnpijxFl9b7KKrqtiVqPAIqfsPPBwy949GH+amhTTjCquyhh2hAfe7jqeAHRjOAyNGo4wtOvihkm/xNGdUnYJqaJ1x5YHb1bQYBknagfWgBwSo0XRqaiPeg+l3s4OoLkGN/WRXB8BdbJArQGuI7yN6CiDGTimsF8Zr+nJgdWtCTO4aIcSqn/g0W/YfDoySYz5RgJXhLt4kEfFHan6EZtU6acjwqCwoDoJTZPYkqfDjVEBnMG34PHLKakYFb0FSR2o3PxrjbtAPQ59IGD6WcdXUg3vAn3NE1bAkwtljjO6dVtXeDxTXIrshinK6BcoUlCfFFr+PnM2UqoToaqhN4QAwNceDKL8iFJN6Bepz7QkEj0e2H+UeJMik2mnnR3j7pmh+LHb5mptTdZzJVi9lstlunH4mBh6tQV1Js3oEsjuv0WReGyvZsFGJuDyQwcLoKG9RZUhzEONo5odfdDzCh0+udQMU1tX1TVqT6SPXIv46TnNt6GTTeQn6R2SwU+bzD/j+PR22vIvXgffx/Ys3H358+/6v0bv3bz++vXz72sBwGuPqFkDYxH88EVYLzQOhXMGYNE1ljQksOvPgB7STNBsuqpbWllWxX6Ze0tGHGhZl0METOA37OSo8O1JOYAB3uXUqVfBplRBJHm+ZmZMGCju0KMJXUcRHerz8AS9m34nYTiIsTdU64xd+LD7KadnH8I3QSLJQnVTttMGj3HKhP86IY52ndVDRiGg02jQidh4aBGvZEI5UAHfRCHCM88KD3nh0aGB9RNy7dDP38Rvw977i3z47G5phc7DKlsyktjsKlSdleoe2R3WEKOeFExCDkzJeG0rw1cY0zl07HKN0sSDD0CE7Tt2sD9SCL4cq4U4bYLLZCjs7ehoQj4Y63ZdicMKWQBU+G1QmrVmvkmSiLLAOJ5KzH2UAQCPRGg9/18lNynY3GRJNbW71/qncF1nhFvQ1SY+U3YxOcxI8zx7xoFoD19/a3MeC/qJnKF0hjwPVbYUm4RGNQLn+NgZH+3URr2Gs8Si9NcTrdAPLqBauaAhTUx1ubptiVWS+I0F4VBhrXHIiSfi8CgRjVF0tR3RaHq2ub6eOmMC67huA0zm/MI/TPuOzhj1Auy3Ts7IsnxGLevo4rvT+p++9y7/8TR3TCxvqsylDj1GliNGW1Kzn00Csi/jfhfc1dbv1dhzSOJdoqTgPvIvAOztmn4ZWxrTa4AcbYEDBnpwnk2f6JZpX0BYSnp2dnQ+RxSuUy3UyUSShnSSAQxa7pk7XCR5TnC8sGdQUZMO9UpoWHjdWcd245AEY49kIYgt9CrxzdZiluXaU5DdpVeTbJG9GuKJXCamQMIT+areOp2c+kQHKQ6NNLNGpIsHj9IQ9R0BzagCtmngF4MzI8XERjCpvHZZ72xNB7+JF7qMqoBTDXda4Rj92mwF9WvkPhPwmwpEDaGgE99U3p5j8cEx81FaE3YOi9BAKnBBfBtT1XAPXhZbx6hNo+3VIn0PTtm2dtJSpKM6uigpGZFtrGHGVRIcKwWSgSrYegHgbV3mEBiANEJTQg/B06V7I3MklkMH1Nq4+HeqmLtCp6rR7qLpTyAURo9V9u8siPoxtNk8veqHEIZcLTbkOLkdAACK9lYVXKURA81ulzpyDyhkdKtQazWUWw5AX1ae6jGFlkRnvCgAVdSirDKGBtvnz969ffIj+9vb9//nw7sXly+jy7ZsfX/3ku7OOS9GntSjr0v4MXAk/+u/lgNBb7+RwFngGexA9tz27LMMQ7Oq1D1hF6l5tr+RNFl+xLY5+CZcR36XEYw0KHZVC3zbNovIJvf6EvAhU0JEPS7iKsz3MOTmm0GDqH6zy4yN55JA/S4rTg79glMusQE8i2y2EucJc2kD7VdfhSK9IKWXUT5pKoiTc5yobJvBncgczQ4wSyV+VO980jPoa1+cyI+aLPogSuyK0Yj1CM+v0Km2gBbL+y+tWb5RjxVW5m1koIslxBdo5jNbjQ94MUBGEfBNnEflkHAACqkMJG780qQFUyOW3ybao9kNwYVvMznz7GZ2R9MM2pQD2uL27hT4qi3tckrOZOSuPyHFthHogbWwDcaOjHbeyxqPRAvSsBv0NGi427Nghaz4iPUHgSpvRJ2CHVwAZH46AUeimoLmsEoUNbFUWZv9tDgC4sFh1vDrDKqzKoaUdjxetBTghbWXt+FCK7f+d3nfCl+2uZPs0ahz1tbdMoAY0AmxMr/TrJIOZDb2XQPh7GfdkrU4OxRlzj3rE6jqBgVl78RVsNwBo2tSiZOBwA68oik+wH0vQ0F/tcna/gBXMex1SuhTCuGTwHJTPIZTByjqHP2XAMg6gqVy++7lluW/L/FWxo0MWOkiyyEBWMM+CMcSDujFqTZF9zIqwlE8D7IRSPGfFhxRNi/x5Q49oHZDqRxQ09P5jV8tTaVjVVO4F/HYdUecD86zcPQIe74imH57kvTyjuvMQ0P/xWH2+OJEz041UwwkaseOokDUTM5Fiy3WKXGzYZW1GxUDXto2mtO2uUzrvB4amtGle4bzFJ92WP2jt/Zn1zlbhaURdQyuP4UfYEHQ3monxYf0eVXpLNtEAe3FDpZjqqPf+2EZfUKD9BrbPeIuQPoLH64KWgdUmOfwZu4AncOzma7t9axKVBZn5ozUw1l6zTWgtg7RDNfMU5RoDgt+oOsXAIsfcN/gLkgn+SzITn9gZQROKoqhBUglwSwFt9bnc2dtviz64Qg998AdDHx2easp06EU5ITnfL6YL3EddjB9LTS4ltR3J0VNnda2c1pgrEYlZ8+kuQ2dGbdS+fEYVEjypDOsLJ1VY/syVaqEyPUbCkVGo8qQbViE+KuQhPxP52a1IlWyy4WpNtTedctvGhTMyY0GmIdmH6C3lljxIW1NIygCyORT4KpBAeP0MrdWjkk2YJQ6tLQOAHT554spwRmCDrjyZjSqNF3RzW+iDAz0M7ms5ebJXMyl/JDTwi0sX7jfpzLiH+TGpvod36bZDreTQmO37tAKRLbZSYFucla8xcSoQ7Tewn4OhFjp2Ai/ck8hSjohoSw0D+zktHQGrvtkCEMZGDotISaWjInOogp0m31z9pm+xuiNAFi9BAuuesn/1Nq1phardi2wujC1dn08xItDNumvOP9ToX4mBm+0O+46IV8SKR90ZOatR9lWFobSbDmmPMbx+bbQZiGWgfRY8Yqw0tWxgmjz2Isom7CxyCF3nveJh3LyrnaCakU69e5/F6rQrgNClTziX/dniZg8dDvjQsjkp46CD1tjoRK4qROf27nJDfYX3DaCPpps97xUADaYOe9PQeyLaducZUPb7nCPcDUffnmIAoLPLEL7vYGeWLxfy/i+fWnU3Fu+pt+geLPZgFZfAIT+siaNiT0XYjUYJDWDFUJTDmxAcxRppt2U8siLfepsFWX7MofcWLQV1fFUlCdoGMbqIKtRiBV6n2k8dJgXUcnEAqpuiNGbAdQKEq93GiXmJkUBJbndjQD2L6EyHHSt5NOc+i23jW2MVBE1/8IijrV5SJeX5o441Wq4yzByPOTuo0g+2SHFBaNce9XqsTh7ccmFa17sl7msGW0TKfhjcq3xQ+zHugernLk//sYNuJ9slnjMUG/bjZ214VVzDBCqvJ0Ox4fbTOgXBHAMxNezajO7psE6i4pPl6cwE4FpsQXdsdjV7QKAFAgipJkvQqkrQlSvaNWgyVMFdYV7cjlR8Vwjf0IRSoOU2dt0z61WVlsQ6Lr75Fu1jVmgTSa2IfkfRGE3ChrojS2pOrTWI03FvRcn4loVCWbV9XqhCjATXWM396QFrOuo0XIPNfmLSC4xh3fG+tDbsEXpCAlw8TgjcL5XzhX2OI5zkiCSp8ve2zJI0t7RSMHIItRegIeyDWULo06kfrLrEbXAKf2zFITKvidiEWLNQPUVbFTRwl65gsuPyGjgDA7AgWvMRV026AX2ojiRYEV1LJYrHFw6nnMyZtfGTZXQUecKb567muqET0phC+sQT6yywh8B0uiWX0d8FhT8UmxOEKcP52qqyaNVgfjEf8tgJ+rwFzeLGVl2oxvAxT1nqBtqViky2aD3EauODvlAWmxJ12sXbjdntldAt6QzYHLTmdeQiSurAE1WTpWpLoAYiMQNtinE9vmjA07UYh5jVzDWBLNzSir/BgN23hqLPVVORFsDn7/ppQGfwNe+BCvJ78TA0jR2ktf+iLEKMWruL0HM+Io9fvSDTdRcPvaSRJPuq0gfkDqtmh6Zhe53jGBLB9MCl6YisiY+Wq+tb4m8WLfgOMagizss+2Dwx6LLPv5ANKuUSxFCcKUMetcdqpTu3Qv+4eAPRrcfuSsDNTUl9dzXP1uq2fRXc02xaQa6SvTgUU+WgxqFIM3aNt8+JBU86Px53K8KAU3yDjoawQkhk+9ByZf8W/dA1+iBCcOVL+3TUgLpkpx1ZqNRfVKsUbI5ZeTQQI4d43yYdVpFY7PMPXWWnf24k0MgaD/9nxu//Wc+oKLEmbVjIM/YyTcYhtwuC1WKQC10u0MMlW3wASxzjBBYXwOKKDwR6eR5amF+nBwAeZwWoS+FX1K5TYblu0UOwXT3IIu+pnp2eWBg36oo2I53qcybdGyIhh/LHhpRo7d7gYmWaODSmEs4T2ZTGrjEJ6220XhyVCLDQqk+HQB8zGkJbQkrTAwR2bFDaUPQbS0vTOo394gBgNACZYJvuysO1ZoM5hKCW/RuMNIq3BxZAV0146MBTUkyUMS3EaKl1VyCQA9kIFJefngwctNXzPqzQcEou/gyBrRNtAhRWJqGQh6X2f3B5Efxhaava78rBwxD/g/L4C1fcw4lLBOrgmjd5M0Wt5OwFgg7DJfiNvXVtkY+wB38Tw3Zt7R/CUdkJaMM4+9LtovuPPHooJIo19Hpmq5QascPVGVdVGSMXybT4qO6OT9oL6cjYI/WjdHawwbE9Wu8gEiftFfoYaOKnSzYq9BbfxLusmRlHawFlncmy1cPlJ8cNn2+gEbRj6RNlNBeJUes2oaNvMWKKuZgFrWsxTu5WSdl4L+kPHe/V+G56hIApEgO25r8ntRFM8ryekQYHaBhn8IA/z9BIgB+CxxKndfZiEmpYNhvvVEslhseW7hDL+YF0f+wakE5+F3BkqGdsxfhDp31alA2dkWx8ynm1y7IJG+m4TW2CZIuisldZNksk9Xtp9qvWAv0KdmX867nnD/l7bEBMJt692+2vDnX7qwdyj0JZBbIeCRGPxYsM9iBoT8L3d2JQVZmfQtctUy0TNX5i62aTbbShLFwjFQLpGqZ77MnrMiWtxcSnYuyHZURGqAA8FhPx6TYuPfJ2r9VRO559JzWuOTrU2SH2lExhE39KvB/evbJ623bcNO5aKk2CPIY72AP4L66uVBhAu3xY7vEXLtESHYXI8wjjpmj/PmetkdzJdDgo0Fgy74RZUUzcp2Tf4m9UH16zQ5oKwwrsmKzxotUq/aKK/Auqykdomn5J8B+M0javhUXjSqWP476q6rSSkqvNsK+hzDH8QU1hNlLgnnjPQrFRSV14dRH+15n3tXcW/ukb4BNAk0AEM58zuFFaOGO5pdNeBo4RbDA+6RpjWkYWxEAhz7WA3TVsMqF0TX68XAExXqkzconeY8UpMF0zu24B28oBRU2oWirIuFuvtTVP85y6QF2ZW+3a8BZhvVvqnrHD9i35XM7OwrOn41Zo+qZxxwQqI9mNqC1M/3LW2j9y3prhOueLbjNhuq2vQVJIb+d8yIYu+Lgv5LRiRUasYeZjSH1S254tuukOHBVg9FgoOAGYmIQIHzELGHCPHQSLhWifxB9RsdmM2v1qmKhyzMEktOIW0WiWcc4BqlLLDd0lkYBu5RIvT9qirtu7pPWaFVXxlV633bHHPjRpk8Gi8kf3jMDD2Lvndh+mJl791/zeybM2V0h8ZcJsv1osHgZFhv9r/oIUfsdNKYChzxta0n8KUFrNfESud5JthHV0krVxPIym2ZgeRRNE2695idsoq7a1f/pqcQDsNHy6eXD6Y3MxWIAl447H1VlR4aHA3mSv/DV/k9z2Jqv02UNPmCf6VODU9u9de0FPMSNmP2gL2/OninsVTL4giHi62gFe6MZL6SAnkg6SpzL8NW8PbDs917vLF5iwCTMvZtmOcixgfJSSVcGvFFO4inc1OmBDX+Kcw6XCNuRHDIn/fQH6j2jDyB1IblOKzsdkvfyPddI72Mc2vdwRaY14RrqEVWJKzXK90b5SSg/b+FbmNYIW3YPK8wD/D6GSH7TKd4qHANV3kIlvEvg5oiYBzAz+DwiB/IBZAJ3cv70GUdjqgKqz3hyvgxKeTaBc21H+5tBsiJsA6i790s4OKsck67m89a97g75vE2QitXrUcWr8Xo5HKXpg0NO5m/jE2C+MUmn8bUD3+0WQtM9Q0RrFcUicMlEd/6u0cLFwSVKUd+ySd/nzDy/CVioP6dWXBFZLFb0Fsc+D1TcnFNYdocecpAsYb10QkmyAY7VZoi9U2thTa2eCyR1bceM42AMNfrRdYzGxQJW7oRuy4QXBnmzLZu9qaMo9wN7qkkfruscMbhu8A23LpmBqnWki8GwTaMfy9uB6aElTE8QhxISS/X5hTnTfO318SB3I4tWnGiS2JFvpAzl+aHmGodXQyY4huR/4uPcZh+i1rbntQkcQ1f7f6JElnljWrDDQA85Y/975EvTQde1yfMnsGNZWeOLg+dfhDjrOQCZ+gE2s7shLlLHb2T97Z3YIsvvxO+/cRlmlhmp1cIehet65dVZ0FPuXfVhji8r5xc4DSQE/O0w9gG6Ybp/cTG6tMaQ8MG1KOobaB8q/bA6wmHhIM8f9G3rj5g3bNjAi45B/H3tsYvzDsHfBwXwlPcyRYZpuqROso6v2AyPD/B65nPbjxLwZAOorB4WvFmrNcnAzsko++dcpIxxvEjbjqV0iHVjan9u5QvzG7H80j2c3OcyfjF9PMVZryKvKtGayKhzzwnLzK1CDw6HfbcsVcjzG0wn+3gy72rBHjcP4QQHieNy1cvDlPFWmV65vrzPb2AtV3XqruGY7EZIpan85tgpc9yGMEKptGlLhe9ra5/8n3fxY1VI5BCOtLY6e9DuT/g6aVlfL6jHjpdur+jPpXAfhtOKzz3xj/IPnb8/ODtbs9YNz3E8BAEXEH4KgVJyo2uWMcqtiW3OE908vurHjl+T+Bxoh7Pg3Ewxb55sNnEsL8GYEXg5xhgnDdeA4slCYIRUw9x7WEEe+Y/49Z70GnlJwAmF7gWdoloMC5XePD/0oZmEPKiB5v+Lc4eqyU73TUhxj5mDQ1vImlWTEXX1NEyEZwjGfuLZ2YN4ulIfLxM76B/yZMsu4vrE0Hu9evVZck/IX/SZLLEEyn1V92sngZq8/Ro9qWWG9qgTuwU/UggV6VNrV6CkQJFLloL/qz3VCKZB5dw6ri490lBUaPVUBDjIkgITgDXOwPVXJK7WWgF7UPeyPaNhER9Zx737iJVVV0u0t1QOarUjp2avjL9K4n1O+55hMELrE1Dk5UMLuEXs5zVsOcRX2ZHf8FB0vZfTE68lk6Ww6tH/7YDZlK+GiCo72x3pS7bVvHwdaJt9Doh13gOR7oH1kA+/n1x/fv3j994+vLj9Ev7x8/+HV2zeOFnY4rtuJKifVpQfW0ZPGDRGd5f/8s0XU9z0QHzzb39doVOzcGGmHRrR8dvo7skcvEB4+aweEk6xjIHhmZvfzqPqrUxthnC8lHaBTUgrPVdfItAxm5LNRFqnuDpqchkjkhVJM9AmppgpPqEJt2J5LiqCMw5xqlfFG54gnPfc6XaYmObdKmkXBYZzJhqzT7FOhU2rRDwyo0xEoJsEifRuPF0RCJniRXnOiffE4IC8nLq0z1NjtwPhbbH4wPc2lHkbR7N24JuAMNmTFGv6Igd6cQcVK9dN7yYt4jKLdf5XtePgaMgBWDce/cRGWFOxICwNF3uqkUMxwT0O/xi3aC0B3zzbq3oR4d4XELY8nfYH10AxBs7gRpgwC+bxVUNBKpn7Dml0Ca7LzaWPH8WIXbbrc5UwfivQ5nBC+/vjq/YeP3uXbv757/fLjS0em4mUgoQKH5zHu+sXI+drblUD+KhtGnNmxnypABsdR3bCgzBAK7nsJlJSweIofxLg8vgzkOm54cSGmIrjkvLqU7Z1yVQhFt4Be066LXcSbavTFbuJjd/sxn1qe6Lxr6QustFrGKEo7etJxwrZy69lgiioUH21fmdidBHsujAOJ9lwvj1aqPQ7MTTFslxIfvdt/RIXDmhPViCyeJfLs2sK35Ux+AFGuJRiqOE5+SxHHZJpELw1+N5+cL6zMfciSLzlycO8/MtizzXHdwEMeMcBbbs3Q0CUClzzl0b9W0LlYWO9dhkiEwQZlDvpGeG7ULn6l3zvUEchVUxWtBebCjmGym+HQ1yOTWuwyDrPkOCNMNon3sECHaIalOyo+mzrl9JSa0oHK3LBKWiDvKNp/5m38e054SG9xjrgnUfQQtr/gnoJnUE7JrHRpOrOaHsz+9Jut5WASQqrQW2SJ1gme6itwg6QCAWD16MEb3VudeRj7NgxfLj2j+84K2GPFjz2wklQPNWawljQ9vAh+c0AXOSqTfvk4FygnGicQZhjpsLD+IM6WGtUxZQEgpbweCx3RoKy+Bv2bXiWJ3hTNtWgj7HcD3UWaRoM7JlNh90wVKKeVn2u5+Uw2c8Lf8eC7jmBoO6dXfChe+61yMhHqYIjbDzxl5ZmpJCDKzEN6RU/sjxkgmrLZubXD+V2Mhlbg8LCr7smBiKK7wL6hxYBZ9xjCzSc+Tnde6ahk09TvE8SBPtTG4dHB8JhvpGGKlHTQTbt4Fxh+Yw8ochwcORh6556GwCtXcSQXXjzutiknoRTJT2ckyEIQIgDJNarbpnh+qNB2Lr1a6juAAs88nS1A1WNo75Mad4roCPb9q9ev3rx88X7w0po/ev+dVIW5/QhYJK64LWbm3eVkcsAtQ154FBDIl02aLtJlTdfFrahm+O8aCCHiAdcWi7kPM7Ys/MVIBmI8D8Mw8KZPFz13QRh2ISx3Rn1lF3I6dJGJmNu3jSx4+zA6C789857QlKNT1jN8UDhZJvsmoXtVmAFE92whPrtYP0RsJFfmqa8W8+n5he3H0Z42wJwkBLb4xLv45pux1acdfPnTeEwH1COLEeHB+T0i8cCmBzorHx9tQ43Ib2lH6rZawiteoM7nCDc1Fd0BeqA+YVKHefnZH07PZTnGz2CViVphL5KZffXUICxagytrGc5aq/JkOE+YWlEze2HjbyHEmfwdhuOwq5l78uQmW+GJo0sY1QlOjPvO9Ap3g63rEjjRh4TTc04bKRgaaxKzD0oGOGvfJGXcslQssEiinlT3fK7Q9uNq3930+0Zj+aqLh4KxrCGIbtN1cx1dk4UNo7HMpxZQ62xj2j0HUafkpNFDActYMO8UbuNrHR5yqFErdYcKyFOH732NDFTrNucGy7jVdMTXvxMgdkRIsvwy4PnSJqtAh0BQOYSeLyWo6/5TsjeJSObo5us49p48NtLnaHxPb8hscAi+iuQJDgbq9MfmjFtXcvqOydca75YtuEWivI+gXQbmKjW7jMD9GMm9oNbGoz1/Hew5hIgm+87cMNoqx8g6F67etUnDcOEuZItDmzbcCj1N2MpPm9ZXmM5Xie7WR0YbJdNUyeT6tCMf8cpeFmIHitoirudWOpZgEUqwbt1e8fbQYsHmVhJ6tDk0bxKsq2/o7NdyMp56lOdZuwtPlX4THGEED8rVjQ1UbhyC2Z04m6t1mVqn0ISs4+dN59DcpUAj7+RzxW+203Dbj1gcAbThnUKXD6etfkwqEmVXpaUuN6Pje/RWTfq8TfX96bgRHLqK3U5y4WY4kb10YFr//5L75KTFX9ijDhkEVFR+dsIvVIlHsyCfxBJbTlzh1MmNYj0dSm9MruWuwQrvyt5i4mj0+QpV/mNzy7HK2+wfTdzSthD4dNujxGhL6NiU/OOQcBD04XCnjqSiNAx9s/HYCKo2D+sEgeGitq0p854iiy8SyF2Ix3a5LfBOhGvnQMCstlYrKibMzrXjCKoS1IvVntLN0MFGU5STLLlJMtdu6KXbLUw0rB1YjGLLp+RSYkHFvRhsKMViIwVu0npnLpo4gMFvl6NIezB6q4Rur0N66tNSsWtvN7LFzay7IC4d27DOAhzwIYmdExB1/Zp8yiQRoMXUO0qsslTiFFQ8IdWAPdO+Skcv7dabNrk6JlO6eWPKV5VYqqVbCG/OmNLFJQfUSSWAVwXIB7xBk/Dw7WiLCR/niN2Tk1waM+lBJ3hy/SPTqSem0+d8JxImQHeu49BHp+u0Jg7R1mugNhbERNpxjhqjbCZWVVGWeNZbVCCirtHgiJwPVIwID6PEVq0THbWg8hmcpEdqzBFyt7y9iIypCDjmdbHukh9za+XTb+4EvqridYqW4GhXW+mX+tQqX8xmdEM4yEJZkhN1d7j3r3+qmHN9i4y6X3dFF5GjyIR5at1E7nc3LLLxxkYvVU3bhChF6IS7KY7dN+73XUBsrvfGVl7LwTTtr69IeAM0fbc5MNk8bXZrOizZwqrV3mTPPfficrqdrjb3lvtDy4X6l6Wl13dFudxkjufWN3j9B96oKUxQuPVzfWEzxWcokxrW6goqZbLDNr8HfoFhY24oGbbIeDtraw1aSV7THT/tcdwWbEEn/anwfop3wMPQ3GmTcYDnakSgE13hOS+vpcJD1phnodnPErAlMvOx3Q/7jae8dDVK4D07+19khsO3X3vfwlPavVfFCdZq3XOdbtNG7jOChl7Z4S6KaGEK0CufLhqXI/8Jchvk3X1RMBQDZtiK+ASpcYH5hpWkMrLTmf4KgyZgUHJ3FOzlri0sPev8YK43iZuLKFst8t/ddtRzr2T7ck7nyHnc2VI9BuofvhBqxfyxfauum1fOMRgN37IrR/xQ5NjVtbyCkBG4jUnyMnXVNPoFBu4FtpQOt/fmXj1ZYWf9qOwIavA42p82TLQTXtYoOPjqXllj7C5rrsMFoQd4x/ouX+WDwrkF+NpfvKi79hjaAYo6lCSx973kS9QHW1NP7/J8Cc+ifSRugMntwMrXdzjFgJFhpqW+HAPtetIiWodwkJu7xh+H3EyDu4qT4WBCVy+0rt8a2uphMt7aSijgG2cqb4RaBzMECqZXdFYn8Hg2Dr1LdUezfT+zc9sxNu+3LuXWwVUH1gHfNs12rSP3NIc28FfmzuhefESuJqRLQfeOXQFtgW4tKc6PuCtRzlBiY9UGHf0cXjfeLWi8dCGrDRxdbvEOGkpzimSohYjlksNjxqfpwJpKWK3otVZYlwuw6oieQBbweNcUoLrA8D4mstIoleY8XunMzoComLjY9nNSKhOJMrEt2ZV+4G4RDdr+yQhECSVbN6I8KcVSfKrT3LhaqV2EBbykOZP7xFQ57Xqmb0Rm0lMph2vbcdK6zssG/AJTm0v82kTOzLvxRCqri9xM0slb3J5wUnPUClGeWyCE0pJa2ZZFjjRqtCiT5jauG2s7acFt3RggO0xrV9ly28G9JXl87wmTlRssawG23dlN5CypBAciZgPtMozXN9OVqLby4K4u5PE3vGIwvY24BnZUiNBTwTrsjZNS4kWM2Um0dwVpIxZo2u0GrQ0RUXmxaW6x1+IaQyOgQyCF1uIVzkSc7ynsE/34McjYCBvAuEBcZ/6u2Uz+pC5P0dk0lPWv54Qp8I4EMTpHQR27vX1o456uBN6T+cYvI7q19MHX/lQth/9u0rqW6b5lMfD7bea9Vu4uaGOwbhmlg46pmS2lxFMG5CrxyXBV34CEpONDH08oUNABr5j5ftCdG1ztsLVZZ5ZPHwlX9HMBSOEP6ar5G70YcbkAFmaSrclWMHPSjJi6LJ9xJSXVyL1WQ44hKXMVKRXueeYGZh9Nu3QiJAbk9nmQIqV0o95wwYcOJJUD6t6a+6mxS7sOJNrpONCkYbxw7LvdW0lL7S6DujvChm23LMpdTLdf8JUK5kpraybHY9e6aC7vbiX9IDWArs+gEFrb86u6yorlyH/ic7JPak6HWR9Tz1Su5ohZh9LMpAuuJxyHPN7b6uyD6FPKlWJtaWBa1wDB3BdHoS8TsxIlkZO9xFXoNNK9YfLG9968HDiItzReU8G6cVsSLlFmfzuci/7oEKcXK8pBRrtc4y5v+TArp7L0zlIuRSNBRkvKoo6NVyB4szNrI0nzI6cUkc6qTzPkVIea+DLEwatHDtAQY6xYYe7wAOUQuFvvo3pVlJKYP6mAgTbWue7IGWoKzFQvUMYpknHa1e65Gh3OlC6mYrmNT53ZtK/vdevIMEZaJ43EB5KAoNVTNPcvhoPTwWDOhqvmkY5Sw1gP6cDFcPzT5QEC6dtzMGHYsRZqkezwmuXODNi9xmesH50pV1iOrVFUhSDwpGKX9xGRDtF0iMiBcZyIcKJ1SM/ARNPVUE65x05lC7qZuO5H+wRr3Ir5GAar/dE7kTqDE/3OWec6eMYEv7azSxzYSsIM6TC+JPk8xBasV1zWmlF+4c6nBfD4bOozQiR1lGIj2O/UHKaw5ZRr9lQFPHMX3+pH6F6eNN9gFKuf0LEI6KjwJlqe+eN+Jk0Z/U3YzzciuPisRyEypxyZRDELCqUgOrAT7ctg0GyKjPLR3XbsOMFbUGUh05XrUKn9xR0KKGQ3YNaTwxQGg61g/wStOQRgR1uZzAZqYy7ad6+0YWRM2KSNnMOjlTIq91rZrK2/3CPjsbnBUyWkJgKa8jUAONmPSBi835vJ5THCxUF73o/yYjD/ihpiwkpdfAvb45V9TY6JocXzRSC3qk1Yc01UC5VjlamRcpo5TBnFvaZQ/HymSXBkaNDcPsbP87PF3LI6RXYwv+InvQXajM0CZzN+B6A9DQ7DHz+K3VnGMSMwKQJGMb4jA211Q5u7ZnxBWV9frTuCemrS9b9ysDOibCR+GPq0b+gpjKmbBzP2/Jyjb71Vd+JY5Awto2pssDWnxbCx8Ck6D/kTDPa3z84eM6ikjokkIVawcwNaAcyEMnx67sG00tS1o8Rh2XFAd1AZNyL7sijcVCo1oQJBqMC2Ggo57kSDkNKumoFvOlqGBvooJUPBZTnfWQXjo8tkkK/15GVSaIsIXyZZgSY1tjTqYO1OzC3t2NtjRFvzL96Vo0UAT1OUm6/anL/nXTYXV9czczkrugRqs1BlILaTVqbWGnwczi3DG76MFw2N8VNKI6JeyeWfnK/F/XKuMzL1Bt4emgh1oyymePkz6XPfnf6ZLELfoYOeHUNI5mdLq+jPBCGXY1BX7UOscWDet1P9u4NgarVUypBcT0d2n4/mNXrvmtfdPDHUz8mmyDDgmO1YDiYoVDGGnmmaGo5hDRV1ejcah0RUyUgvnfA6uRMPS84IFDtO10fzR5GN2BgZzUVzrM5SrC0LzD4GqajSXIvTcRN30G85iMePuK7Dvq8jdl3EHW9vmzBcZ+12wpCeOxv6r8bA9iwTkyiifQZGsR3pu6SO5iA6aUVB3ZeOCcm2iIyt8Tvp06t7EDL3sfZnXYIfWGscim2JIz+c2/DMJXjn35ypuFvUofmDvh5PyomCbWqpzj02b17QiUDlAy1h2nIYAftrREfSaXb303n5+ZAMQ6Yd1TvYr9yNfHLmtZh5XpLMGqn6ASa0L26jMl19ylQoPPJvtt66ty7zu7mzM11oL6mhLapocwqAWQtO7blzv1UfJR2Ap9ZKB5pWEx8Pax/hCuuB5TgNHIen8t6Rb3iCpy16/Dqhf1ZE3LzqfG81Nh5/Sb65yw+/kA7/5t1/t7NXWUfmmH9R2ZV4KzD12pvjlr1p6vUmc+pa+vr9T10zWy+kjnGgH5JlE+iH09YGI9A9+kG1ltIjweFa6gWnFtkAHKU69ta39cp+GH17ounAhqgfkb596PToRrUP1pMnlh22tzF4Xyk9nv1orCNGcXeO+Nqjm7TY1dneslnKNaC9gGG9EbgPaIJEiqcjLjoCZDM2rJ94fYPkTif6Sxj+iT4DJsOlcVxy7gumrDhqHFXgt1k8YvJHPwJY4Vc3U3PFgZuArJWpFrSsms6poBL9Dl9UVzt0THtHX0brhN3pMeQvitbFKorGVk1K+x5LlZE/mYgEnKAExAXLb9QjQGtmvi0l4SUFxcqVxCJgZu71nO1/10lWzjAvKm6VMF1kEucqMt6TZGniYpDmp3RkfErTwXkOBzug6GxyBLtBIIZr/Ftg+KDHGkt5sU4rPZjWYdBvbUZfuapurdFXrw7WKlPVIGbjwxxbeB3hbK4Smi0MQExvNgSL/CYmHElmgVTVn14co4WX5JRppTSUwyNj6tu0syvCAsQcBl6zyxPOgYOfkuFOK+eDCR5b+Ry0jzcM1E1RJSy0j6H6C6V5UPhIUImT6rE+tR3JoCHiFT+9+/mUbRnKSQUzHvDZ4jDScis27mUm/+aaeyuBsSbVwzoxOd4QWLyeMMqyDmuTsumUPcXor7jTC+KSkEnwpz/Yg5p4mW1JjLBBVq+JZ9Vh+31IWfFukpGJu1I6uxhoJfdaWO7JH6IElCKDkbxW/N4trB0F8NnNF4qq1qiDpFL9e1J1D6XrPgTHzZnXLjWiAXncCWO77nDgNc9Q+PgzVTkP3cPA0RF6DvPa4L3CeIjfbnusj+eGRJxKG9c50T5xcHTTdnRHJOjtSlsZnfs2klqDtwPmcELaSmD/rLlEeNLVe343inzodEOUGTcmDjsyEBWngDTFqsjUDq87kpahUhc211XzYFhWS87SbWD2rQlVwDVtutWO2jYtbrHUd6MyBDZ2aiRhAO8fxvzSIgjODmh/tIK5HPOR04yVNot8PKxvtH+jEbTSvx61a5nbpshp05JuA7mQTQfCXqxt4jg4bYcIxZmJfpimUYn2ISBHKMTTb8PBbvQkYXxE2+jUnK7Sxrt8/epUdIrn3hGMdrnJ8dlZVRaSFnYLrzPJnZriSUxObbg/YQzRxUdi90GQ3OJBhQqQSdoZm8mHk8OJnpvzTtRuraLsdCiyWK8/Qk6LD/xkxvFAJtYenso+EP2nSsfEyPCJgmL6rUG0CQ1XFrGG9W5b1qN7K2zZQVyHfyKTzJ04jwEUO1kGhtEdguTECJiHh4DcziK8qEEyi7HxC76K5cuOZse9H9+91s3Ebc/KrGeeRLWc/ab56o9tnj1i+mY0h4fgWtsVxtkJl+dknEeOAI8k/pahkOydeAkNvyjTL8v7PTM/Aye3Lw7oQVCGamcdKj45gTWosyXiKXcU4Y49iiRbIm3fxyf/D7WUhsk='}, 'evaluate_models.py': {'sha256': '8b4b02eb1a8765693c9cefcab5181d9bbda5a99ffc6686b7227cb1b2cb127e59', 'zlib_base64': 'eNrNfWtz28iV6Hf9ig5Suwt4QFiUrXlI4VS8trPXVcnYNTPZ3VtcLgoEQRErEEDwkES7lN9+z6OfAEhpksmtnaqxCHT36dfp8+6D3/7mZd82L9d5+TIr70R96HZV+erM87w/NNXnrJw1fSmyu6Toky6vSrGtGvi/b0RZ7fMyKURb459Zk2zy6qZJ6p1Ii6RtszY6O/uhEvtqkxVim3ddXt4IaNztmqzdVcXGvCzhZSayhy5rEGKXtZ1os05Uado3rdhlTRadvSkK0WR/6aEw24iuSfISG8PwWrHvocU6E2m1r4usy+A3jDMDsHkr6gbHtRdtlzQdjOpDucnqDP4pO5Hvk5tsVmR3MMZ1VXUtwK0FDL/fZy0MLOMaos4akVvtalgM+Ht9Jn9YhWlmhrPL0lsYbHIDg8U5wbpBcZOlVbNpxfpA887LO5hTfpN0VQPDg5U/2zbVXsTxtu/6JotjGERdNZ1IyrLqaBvaszP1rrmpk6bN1HPa3qmfu6TdFflaPf5PW5Xq9z7pdup31apfdZF0sG577j+tiiJLqTc1gE22Tfqi2+Rpx3U2SZd1+T7TFeRzKPDfz7CAXA+WCYeiqn3C7lWvZb+vD7Dmoqy5ctv1m0O8ydr8plQtAAFzBB7DGEtZFsJKwuZuehpkvM+6BKokoals8DYuqvRWgk/z+hABNnR6XusMm8EZqPaIfLLebZElTRkB3CZPdd2mSuOkT+MWNhHmiY+ApXfZ2dnbP7756af3P4mFWHolLGNSeKHwKsDXClD9gA+wVUl7ADRp9/qd87w6+9PHd+//yEAOVVHN58UsLVqsiI8XX+tHOEZl1l2e4+9su81TRER4E6/PAcyHP735t/fx+//8+f0PP334+AMC/OJFdXnjPZ6dncFGinaXXFx+HW/zIvNxf4KrMwH/7aCmxJyIq/gBFdzn3U5UgORUGwew9gLcty03xP+QOqxxqQGrRQ7n2S+S/XqTXIlt1GTJxp+fX7wWLwT+CWDFPS8wjan3qK9x63yCwh03GRyDEop22cMmv4ENghHxHFLYojJGzPZxqzMJLd+KvMUjl8Bx5JJQINZavUmoX+DI+7fBlQMqoIncAibhPKh9BLPZt37weLwDv8jbDjC/BxoUAjZH5SZpmuQQjHtdjnszXa1OdAFQ87LLbrImnAALRXIdToPYFlWCpDcU9GsCEr2XsBAOEowob7dAdLtMvc+KNhM/4CG3WlKZ3J77BmrzLBln8HzKzpAGMN5FcCoAdaP97SZvfH5oFz83ONrsAdY0rm7pMRg2ZPgdcA4fO4k2QEpa31pb6i8k6lx2iwuAV7ZIUhMgAvniDwlMIBRJUVT3cZmU/CIQXwnvv0oPK6fVBpZp4fXddvatFzjTAkorZ9VU962cFf6EA4SY4NNrtRFAualwuNJ/31qMzuQ9DLvM7ou8zBbe1BQGB5am0sCIYTbROzgh/0Ev/C2gRp4VmzIBRrjQ01mer4Jg0Jh3YQenG9oFDimAJojV7ryHLaHU3jJ4DNRCN0i7fWAQgDlZqZbYxlAoEy+pEBcZ/jJScqEHe6o3LXtIUqCNSOPzpPDbPk2ztoXjCiy1CIHblduc2PfiPPruUvYFzPjn+2rWQslGvC2qGgSB2SdgCzBS8fYDYlZa9LjAID70eN4PIk1Q+EE2LreeehCLhTgfUx8PWT8crsy7opMEW5jmwKzuneddfrMzL2DOWYPSArzCEy+nAnjuwQRILOPCc3gD3GtXbeDBGwzfY0qWFPUugf2fi5m1BlQEo4CC8+gcJyE7oWnwGiPTjOp66zOIlwKOl7uqAFI1+0rMGTVwKtjdCCq3cCHjoMbQEdioB4dVOMvKuKDavuSGgb3QXAN+B85y82t8CH7JqmOh7uPU8ksWBovetyCi+IcQZNUM2VS2gVMcSzl68dpFfKDfgMUpoFuHtDxpiclga+g5AOaqW8JCWRUs2FgxFDDkIitvut3CtHjxQlwEEUroSZ35+r01HHWe1kmbSymo5aFX62SdFzkIxIoaDl/Dxo9HHLqDtKor9kSw9PABiFMrAhF4nzz4CZDHhcSydI8kbXJluUJX81g2eXLjp/sABtAdYMZWh21fo8AXojoA0iZ1nO4jUA1UV6HzfK5QMAWG0irwd3Cg/K4ODbiq7xbIg/ui8F+znJCUAOse9ZyFrCZ+A+eMAW7nDqwL2GELHuyxGt9ToP1xk8DqJ0FtK0kP3BsoQiAzwMrAiQEs8Q9yYXcoS0AVDez3FjBVldCI6t8mdY3khSm57mImAYVMd+g3dwAkbgesksaQHTL/dbA8sEAESA/kNT6E6hduoX8Q3y8QZeX+MUI5+LG8CsXF1creOJ5vn+IuLRk6squUmFVS3mC3hlaD3niTddwZEKrU6UzXQnARzBaUQN/REXxuPzGqdEVcC9cMZtuXOWi3snYQYFcXTA/lPlJXvxVvQBhLm0ooEgcyngDWk4FsBjgKW12CongQ1Za0S9LVAfnbOktzYOkbpZ5js2TdorDB6E4nGdUEPSVP7RfQL/UTaFpf3mdAFOEsxWkF3cW0yVCH/oam+TopcGM3sQVHsu0a1KoEmD0floBIJc4q3s5HlbZzKDdQuV511+ASjyrjNjjVJbJwXXdjxhg1sR+q0sSO2IMCtTinlY3XTZ41o3HBX0RAl8AB7jO+84kBUsP46c4XRSOc7jTgkxCnVuIIGH2oZvqkuTAeLTRRWtqXrfdFqr3LdPUod28Ov8cH6jF4BgCzrbiRp+BIbsjgFE9SJpwYlaQGVJExa0JWpustLs7PQUpqs2yzeH0xn3+jWC0IdESCoNNNtY+k4SOG9z5Wliypgm6g3mlWyLNO0DJFxwtF6isgOzQzfMDJEahHTYliM2FrtBZNyh+QqJY3ShtsfZgGE1+YTv45W/CDoU/6fDujtdlxsMwfVse5MRW78j2OnswtOFy5sVJPduV9VIBqR38cVLDWaIlQV4qUakU0UNvewkbgOprV4mEYeiiXczwURAho6nSkC4sqBOmUt/0vfVJ2aBjBFqFYghB8cRmCLPzdN5dMtAkU0QJfEgOHTON/G9iYEmVG5JewiYWPe4INQQsrN1i/7mr54ncgE2ez+evAUhFwpjxIst5YMq36aUuxOH5Let3ljvhpWzpBDE0zmp85MWhFIrtZ3GR1kacAvEWgasRkcJLWV7eKhaBQyRxBPCdQjH9IRlaLEWf7Om+geRFvQK9s8nWPFjqoauoQl0mzpktgHw8xKM84a+8NqHeqsWWvTckwCgt2l8H7vqyArTV3wOqypgH0ANUYLcttBKzT2hNFIhBzkDkCIzqIpMPV2WYp/0UrHokpwCyxi+yO9SPkrnp41wAKIXSsYwqlYzKnnd3nrekMFENWVPUwGIe8Hyo2W1rTMqt8LdgIqcEYhs/wcGwGI1FPlIYqSSUZl5TGsauqNouhfqyN8cckeCP7OJRiLHQZ2u4K+QMSckQWOy4FgXx6YanNCS7mv6OJ6T3ure/9O9t6c3IiKM+CRIh1pg/K5goWtttJ1iZumqqvQf5pMvIp5A0upBS36waUS/pHwUOaoe28WpyTBuBNU0l2swcNA7qSViQprX0gA0FGslhS3CeHdlZmNzDguwywIM1RP0GbTAILcJNLClbViBtoVSBChnqSYNKJ9G7PApszPFg5EvLJQNjm5dY3xaxhldlDl2zRtkMDj1Bl4hKoHdiz5aEfqh7NKQtcC5AJYFnoNTSLdRETZ35kgFJLAPJGFng5tG2RdGVVfs6aigeYFoCBsmFowQQJqKuKBVJCkIYa/H2uYO6gCXZqgC8Rz1j5M6NfmvJVsHKtArqWFn/sdtTBihX6Jr9jv0ELZ7BD2nPH5vyUTFZMr0AUfcj3wGyByJUtbN5d3h1Au5KydopPMzHHZoD9cdMXRMWQQKOLy+Brsq9grwlaQjY92G+Qzauihx2HRQBsQKJDy+LR+BhxNEg8T5/8oXcBpvLJd17QwTWrQFK3WnuAY28EDhkEfLWYMRkcpHkDz6pZZpLeNVCg+9JtFNskPC9jdK3EaQ4w6IRoAwic4DXph+W0sAbUdTE/f55J4YQ1QaKRtnGN7AgjI0LVgH7SlWSgEv4powOqBocxPVwDR1wfhh1NEMFwUpGdFHiAN2TJnjRX3+uA9hTJOkPUNDML7aEjbCX676umhj2qbtDxxIOTR0RCjdAJC1KXv/VQErdFc29SgQ0nleJgLLUHA2kNOpe8BUVH7t2i89U97H1Gm7RcoeB1bhRtMjfss1JaCfZAGPekWpHtC9FlYA7gl3gUXfF17UjZ2M4VSvdJewt92B2CIOdUQYKEJwKr0oZaEjeZM5QoshgaoMlMSCuwxMYr2RpG60Ig9a/WxFbpazW3CkgYLQeuGHsZlSD9BZEAzt4aDisIjKQCrsVLuTgocKF5El76a7LW2kXYrgylKmqwAPV5foV+TjnTeAuavhTo1LulkV1XtshqyuWrlSPCOsX0bvUYjPSKsSKBePMV4IZlj0Jy6sulnE2O64TgjatAxgNESg+ga/YBv5lfEHfmSYNGywv3KEC0SAoQADcseWBEAegS4Xx1DQPbdjNigyAdN2hOmaEfB7AkzepOgOTYSecC0Lj5Ncq9wE0ICIo2dVGBULu5Fu/fvhfKHGN4hkVTWQj2jouD0kmeTRBfQ9RBRqimNGdX3Hd06F/T+Cvlw0PA/gcU6h1qTOZqlBV9pQPbBryx1Pj+AeV1WDGQSnhliYq2pJf98PB6AN5z3Hi2JutymAh0CD8YDw9IFLz+nZgBB/92ohglJ/G9mOPBgxqnRv6hZDXBOoXscp0YJYyGBa3BYtl8Z25ErkstcZ0awFtUagY7u6moSwAM0goaoK7hWbSgYJaolnEwRP6Z8RFDcli1MkL3r2vcxxUwmGoZTZQSxvr4LzMVDXGdu0K6O92BtPM4art0wT/t4jN6O9uYFIxrSz18+wHpCeis22pg4W2yLagCKOLgQMW59+haa3qykZy2Wg3iHdTElsYmvJrmab61lyQUKb5mG6QAAdifNDDAh9oglkl/W+ZLGcQ2yNeS66b7ZUpm9GuxLa13kpkCpe9qLLPqk9TiFneqqSQx+Bo1npLUHsMX9KANS6VHZI60tF5Xw290znhb4gTIM7f4bovvOnzXWVbj8X+epUigmcddXoQMQ/sKACPXsdSMibrQN0zsK5zByR7r+u5UTzX5Pifr6B7K0z2QUdhgEPJHWC4yFiO/NybfiTqqUDF+ZZqGHWOB2X4rMQ+JzNMit6MJEX9bWvrh6sxBNGm9H/fzz8ZYbiOhrP/X0w2kveHpPv5qd9I9p5O/DnqRdg+UZuqKMCyLyQgCy66EfFgg3GtpnRgVX11gMe0RdMVxiEihftSRn3kqjIbBNhYKMkLekIi0AJaJhrvk4af6jUDPawlDAb5lq494RFypY4BazzxkjvNqhN2AtVhHkqRRF/+YU/grnbVH16ZRSivtgYIXyANFQTtm5/C94pYxcJEmf4i7ps9iFGhjTa5Bwy/6PUm66Z55EMY92ucSp6CIILzXv0MxoWFeSaTD7o04itBPKPwyAgO4Uk2Gje8u/4ltYGRFRZYHU9mQqA2YhPHIOcrBHGrLvBJFOCtgOa126KYmmz9g4zWHFxvbC6qeJDu3AOsBf6CRTMWQIgRjDYrET8hQ75ImV7IXSDNNho5weN9maIHpsuIQiZ93mRywFI4AHdGCQnyOg6K1ocQJi55JXo/vLQMKnRJy3cWOtjHyPLJK5gNliFPh13E6O8Rp8N8XsLKk6oIuHV3wmeSIgg2aKIDbrmHWaNaW0dvC9TsC2EFHL1+HZNXEetiQPM0gvszusqbt21mD6/yvWFO6hqwBnEeXHPRIsiy2nKVjuTIStstSTgympCgYzEz9HExwHol3vOyEuxhgcMM2WECfO/R9EP6YKUnTNeIOzefj3Y/yFTnUyVuJ9thd3nYV+x/2Sdm3aZPXnRbAlUlrm6Clq9oDOuQtGhEtIUsRaSnPHBVmsQozIQ4fk4cc8BePbmmCUebwDtvY786xXs3PSGhxtr4O8yCjrpSBZDzUtYnskPB/r4DKGiamQ5qFudpXstqpUBWyJ56OTUGYJhRFEzc3AOFU7IEOMJmOLXkqvkDtXJ2gL8DaO9qRWEvU+FABHjVH3cjffHM+nxsVGOPKT9soDfBAaTvzYRXq0irmnYylmQ+6UD/RXEVyN3ULf2VNUhQcsUG3JmlBPaCAsZ6o/Ve7ulWbESJHBN8kZJGTfX0lwbC4r81XOsbe53qh1RjNfJdBVJuoYwswGbjm0gxIhD6GpeUfuMDHjxwGAB8vXY/887faOX+rXclx8nj9zKCAv9F9f9p3D1NtT82SPfcJeejDJ2qtx358mqhcAtecxtoiusyXtyuMtlzD3ydc+xiGr33st9qTbwWmb/KtRPnW9eTfZoeQHZlmQGM3/v82T701nSXMwPXWm+Md73PgGHJjZLCoxC5uNlP4TI9P+fZ/DW+9ZWcgwieeEybwD3Dsv7P84drBr53dmyqT7ve2S9ZF3u5goeq+kPbNv/QgnhXslm+EJTFNedrRwDKQoM0GSbJmbVF831TlDbIfSa6s2lRk15XNUfJVtQ0Fi3GFcQdsguft0zLbJ03M8n53X8UUax6jPlNTe41a1FQ/KYYFIuM+Tjb/0wNJZcqpvBjyaeCO55eOcRUGQ1cA2JvagiyrITH3IOhkm8OgTDSX0vlQlSTP7ku6kLfQzhq6AZCUtyGSO8ekQz3aVkbdFtmbfAqF0wucDgSGtnz5yqVjapT4FgHlpQ/sIlSwHcFC1Y26iq42qJhmjcnb/KafMouF0nQO6weiup6DuVWHdnm8c8fuI/UY9W3me29ubpSJdlg/qg/4C5WQuujk6hW8LXQ5RHZ3LV8/936ItGrjZbIf9I20j3gjrd4dkCDA4yfLa2ze/uS+WUnJbwtU+S7zYX3YVGdt4m/Fm7Tr4bjWoEbhXU3yRUiLb4N3I5HAJB0HYmxYxcbYlk/v/gDle7wjKe7g/FRNZLhTfhNhj/BXrgHIo9o69Or8fFPndKsNBIk6X7win8S6eojzElTCduF1KC16wTMBfj0E+PXfBzCqN1vvCQCw4RGb6QHEcQv3hFF8bntVpY0daXgplvJqE2LCymwRKJj3rCXAGJX6EALXzepNvmc8YoeigUZUM93b88U4VbSwwcDbfo2I2+LYSXTxv46+CcUlCHLkkW7poi4QtCI5oAZgUJPPAQqMDxH0DSPzaXgYVA8StfeveMRh7e7gJC/O8W/ysJhPDY7Cndx7R9OB3E/EeWvz8gPKsbCFNJ5lE6arq2j+T4/eVOd4fTDdU6VgBAmmRpfTQHPBeKIHoBu7ZOEhV80anJv9kFZF1Sy8+x3IO9SV6V58zw/SU/RCRJeX3L23LpL01kN6BIQAN2B+YdHEh6jNOv8BlNzbdqFmHIrD8AXVYGKx4D+y0uDdA/1YeJ+0jZ9tQVBblvyofQ+qpMu7IoOC6n5mLZ7GbsEWqcnF9d4Oa7nHBiZX+zDJm6yLrSlgXFMjb0ovXl3yojcT55aWfJ00fr5HnF4kDyBn75q8vF1E39rityJ5liHNjHV66KYm2X1aFV721PHBw/PqqcPznNsKbsAexmicuTK7G2oXYf8U4j44MDoeLp4Mg5u61XCN0duy9vPuQZwNjg2uim+6ZuSCI8kbjPEown/z57fiC4C/il5tHwO5vqotudlDIf94s5k5YTewJPBU3OubIPKYFPl+4UMDPCHqd3SOjwrviazOlNEJjQ6Zhfu4P6NCif4fB4axHz++FbSQrRl4gbLqxi+qdMEhGhwYYJ/ub5W6qjES17i6ayQUijZ6KlYTVPz6GR6SsxGqrA+TaPLr4PQpZFujpl47hO0Ihvwbeh3gEFqOCEaUwdWPQwjwHMxx3CQyBmTCrL1yqGtKscX+cq6vTSwdj8DKDjFBfNSVLBfDsBIoBLcZIGrlqXlZYa4zMkfDDE2QqHGejFfol5yD/19nYWKP9HFwp/D8EzFxKsZ755Dgdoiv8/Ai1Dg7Pw9fR18H4ZNkGHnGbV5u5OyQKH/OkSuhNcIJ0JtCJqzzc1XPqI7tGcH6E+tk11hZVJ8Cg0DMxxvX9s1ricZOsyUOd7XksKYVnnCovfSgxOD2g9IHkYYs18tx6NfKhNAhICT61bodNhtHhY3aWWade1J4EMzMBaPCw8Z9UtTasFcdLjasz7FfNoZRdBSKALCNUAQIDG8WSxpLSMAHBzJNasKQVxOxhNiJyz0JXdGoNLmC34vo2yHnk0leAJG9cvFlvfyX8l8o+NKfBIHmvKlFxnN6QNlz4c8uaYtpICSavApRODcy0aDYw9A0PKzQOq0wUw1MfLvFhDxknmrt0/fNP5DizKLzV0h0XllE50/onDERJvZaGNrzUQVd6vVQpIf+HbFQ+3QoU4DOIaMyDcHiZtlnkGLUc1NVKp+I806p7G5FNslg8hrM1ED+E7vNS+HpF1grwlwInm4E9SnBRVElm9bXYCinCue/CDS7puYgaPmedOVh5K77TnqI+XqF9BKfivn6WWVborGww4ytRmyRagVn+XGumSZlxf5+qmlFp+FuIyDAZ/SDAg2iceBrTn5DEXRkmOFqaGlUtTgVBUm9l8HJQLWfMfmT5QiWFz1azPcxcS2Wx0LmXqqAbIs6Fv55FL0OdKgaLj75p2FTptISme3RF8Ip3ZMT6cQeYHQvOAvhZtKh6C1dR67CgL705QQuMXQ0Q2Cj+Av+++gNwoo5W5JCRwQESAh/YpVFy8JBW3GAHpzGEdmdRlf+zKb8COpPvpfbsvXeqzxf1t5Qth+8pKPzemE3aiBXIL315aMXTM7APRvu0Ebnw5qIqsnnAvNB9XAkEMc81bGHiOjWo6XlarzKVcMB4C40NJGHYjYPJGrD3J63Oiq+Uy+EHAvpwLKP6fWQe2OOOhDHrgLKG3MaJy8Yz2Zc5TcLiW7DotUzJ/BWrt2GUArocVXetBgWmmjzdYfk4nNWmkmqzqZnZgV3KERV0wBerws91zGVsDVy4SS6MrVHiyfr48ZOAP/Fy2DGTEAwNnaftyDjpzuYpQE8sY3uHjkUe5JgP3sgFuG2xjKx4pJmmUBHRvsrxvmRD8jagyu2iOkXEdmBgdXSLaHxel7JdQ9d2mOK7d1zDjfCWydANsoN3UNQe6beofBRU4Y4zNNiVVGor4sGV8oR/0PxxUs5MijuOwxOVKntorK691V2uwjKgihvK7op2vl0E8rm49PzMBwCA77MpDGZoXTjyR1gTx79xCAH/hWKFy8mU9/5geOgkpuEaYA49aAckpVfD91MmqFNh/dZHalUjUmBhPWgrw5sZJQVqAAgSHJaSOxEhstQlJQJjhJJ30HtGV1Y4ijwx6HURbu8T8p8i7EC9IS8DUVxfhdzwiscvGQ8xyopeUyX47U4fOHUUpcM5VF1qcZETewYdUKrhS1koaCuW9n011TC8YDM82Q19MVWNV4/Gle1ZT5nNEAm1O+hjOTQB+8dj0IvmXG2EnkYU2qSACVoJROZNGSD7fklqcjs1GkqG9mPnFVsq+6AkiUV3et0LfMKbxz4pLzyNc0VGqbo5zD9GN9I4yL0j/AplZfeKQAVBu2tdHY/2dNv0JPPVa8uv/kupNpX312GXP9qfnn+eHJ9t96fy0xdNiG0gL9y52UnyA3ohyLCcGzLOMdjTr9wJdVvTMxILl0UljFaxPpzdjLrmn0SaMWarOA4Xyyxrz8hOnG2v1jdJ+I7LV6ELvWSS2tMpkpyukw+x2+IQb0ayKin5BwaV15q/IPVwFc2Q1IDog6W5yvswtl0vuOvKsxNBcM6uZYKZUaBjSqwaWYVTDd51jTQQAQoS1INuf9eEoiXbNF5YmYYbQFbPU65SfNJWopvfPCDiA5P5qujEzi5MK2FojkosB5NS3eCkqr6jeE1Es308pnudDHh3jOW4V0vQ0xe5vaCcPLcD+8mpj4pnWkSDRKeIbUj+Yw3X8pjOHxZpEbNp+QZw35LcaG0Mhs1AxkDI9bo954YuFq2KNlsfLWewbW1XlQyWFFVgYdGNXjQKigTf9MiEHmTyos8oV1lMS97m7Zy63DmukbU3BTV2vdeeIF9lmmBOZyJXrX9dps/RGTo4h0fpozVxNAZHaszI9Q4xVtkDmVsIHUAthdI3qIOiOImOjBHy06K35ADjpxpV/wXRcADUzAUZCfo2tXQIgpU8dFcPZc2ivxzZt2foMB4n697WlIFkRsjXCzI7zyQPcbCBpVb3kHHDDBJqtUMnk2uVZlLsgf4b4Dq6tMINgnZpr3Me49BpyFTFQwuVgU2UBZvbcDMNfAYHu9wohQoPYd4T6tEI5OQw4w50k0PHJcVUUg+D/iPYn3uAJwhDc6THNjwTD1jhG2ypUC2fjDWU0NkZsMRu4QuFJ2tuJuz+FTTaJER5sZ98EcrGzw5UvROlJsZ9AQEiMC+lPEvtm45MVikrEQIT1B/1W7I3KR5RUnKSM1cbgAz0/CfnMIHWlcayhNjRkPzIJSO+nV8qKhybtDLurCi65QEgwCse8qvw+Bq8mIyEFu609kPbgmPEtWcFiD0hv4NQoTWBWTWIyR0g6AAotxIJylmj6svNS1eLSe6H4UNGyCnRBa71jN2+chOf9LEfcaHSc/xiY03lNsYQ/Rgr/RISe+2KfSVmFgCVMQVT9PjJ5OBPq1XfJTUS3XVSkmuVCgtG/LalmqHZ5+xTIaHkcXFut41ADVRf4WWhS9br445fE1flaB6y3zFIkd+6pru4+OQ5UUUUnqbHRacDB5P7JU4ztU02UNB6JlSUGpxARaL8IVDUpiLvMRUJVJAojg/eZ/OStSxbEYDM1FlZlorOzMBamAKBNt88a1+I8iab1XBGmqeJ5wYNuKCHI1h8fobFwOhtZRK8eNAjDIDVnn/0SUAwtQ+aQ7azMRjkGmxFvaHHiir/Vi55HZm6Nx0yVSRrYWr0FHT5BNfK0Sdy2RLvR+kurEkJF9aHJXwRa0DzrlOAj9dvfO5e3VVYJzyz7iHGzvYwooWBHgrJyyaQwUo6RY5eijdlH3PobVIGg/9hL1UppGS01CrcCUnRGZAdvC4Ufwc5G+XcP+BzMaiTyi/5gtNgU5jKiubOEkcCl2riNvNsG3bbfzNptpiAtIBgO/nDgiZcmfUef5U35TiaqIhUZ9TDcd2SeWTnLXDK5nSSNkiieIrnZaniQh2ey1de5gAjiW10bdd2BW4UbcI0RgMw7Eg8TrShZ3Icy3ITn4VDGxW0jh9qgXDFdRnW6I3zU2Ptr1PVOJvMr5SiJGLcbyp0jgOrJaoOMaJbOJ7sxkSuBkSOHRxo+Txib9IIGUFK1blCABl2Pu7gChm6rbfZUW98N4xKSEXkB6t4Mh1pMaOqTRK2zvvZFdV39V9N9vkzd862E12l6cYjSSJ3MJL+01ydX663zXKCDPcbNUvJeZTMF5dnG6t7s/MrMs4U3DwIqE0CTQ3FKHEAOkPgmx9bfTFp8gkJTGQxe/EHOCcco7/GZ6TDo4b5hnCypN5J4k80lkia75Bfsub79MweFti2Bb0IidlUhzavB24koMJV/GU3cC4EO2Di1/g2fd4ASiLBI0fzcuCexbQM10d4HQoCSZSwlnkKEyThER5UYwrw3bnA9mn2A7boT8M+6BZToR0aDlyccx5QS0tMwI962Y4WKdCpI+GN3E2pMeD2SONbHnE27ByJHP1UqOWtWPPvUzC/SmfysS2W59b4rpWBAHJ+7r9BCIAcb/Lq7513fl2m0lnPqlW3HI57XojbUGt1ZEanEFHQlH+NqehfvmkWRF0OJgb+kos7O0x4/px/zc6/k4iNfOilAgKnUispY6ZwmU3GZH18R1rFRWuq/ykn7JGWWsxBuNKpj/gL6eR11jzUj1aMyyTaBUmuJEggdBv2V7DGQ7Etm/w/rIOsaEegAHTGk1EgzD7bCP5FZL01sG5UeCUAmF/7os+9gaqh0FA+iqVdueM/bOnMSQUR47Z1cQhc3y2IwQ64b3VFlc96YmDcuzrZr5uFArz8ayjODFZm8yytAKx3Cz1aThWQzgdKitMhKIttobFkslCQlHdgYKTZ/d0rzek5I3/sLCnSXR4RuwTBhrIEzZFx54BgXQJ6+6yvnyh04osxtkYf4tJQRSr2WHOX+lGhyXjGLQMfh0cqqC+ZJjldIKkA7M1nvdodLvJcnI6vs2xnaRu0Aaw9d5LRIIuv9DkHzmVAc+cAS6kdonBn0Xf7gaB+wMnroMueKEzHHA5pY+HggSrmIJJWaDRzygVoajG7/n30R6fYcofMuKBPX8MOt8MTH6oPFqWqqHyyLY/EKLGoLTjdgBNc6BfAOswBmObe4+AGoU8qcQMNiiARdaf9NFbjewqxyCrD8p9/Xpid/RRe66UMfpMGH1ozTqydLuSkdHa6FaKR+aja06a1pruaH4mgRSOKDCr54As688AUu34gn3ytGML3tBQHCiV0mJ4RXNRh8IK3FpYi6yzMpktnrg5SARmSWNCSc6nvnWn0NsEum7l2V8sOKzhajIrm02jjqaCDyabWuzDWT3PZEuS6Z+BbmqAzIStbKLBNM0Kp2aJ2MaLMf1lCWN8wdPl5DK1egxPKEs5m8SsL5sdyWsqvhL++XiVZWqSiU1Ucoz61snIKqR5lxtONzQWTUXN2Y4TDom0pGb/aDRv4EjRbvRk8HjsBI533JwXyfnVJss5jyFReg+6JyPzEujrKe3gfore6KfPv9Uu/oLQHyUZkBkC5B2X6dEo05vOiKmHo1KfrY7jHIuURvJ5bqjkcG+d9G1jq+AweeixTMqycDKNsinjSzFcqFN/yGL5vJpAAiXT/RpzfPHiy+3V3cgGexvend4BPHe3LNXopFDhsXRQ4cSnp0KTACocfEMqtD8SFY6/5RQOv8EUut9SWj0+TlNjRVam0Wcy24Q+QaE4gvOytjdNoqVYvnTs5sEpFmISBVl3FEwK2YUBOfwgMcmVgZW+VadCoY3a8ykbi/vkY6RAexugJaBSuiu7ZwN50K0V+I65l5WZP6bM276exTIHrCeoxollrk+/Cp7ydnufkryhFIRmtcylhQ/vrIXFkDiZS1oZl9GsjKIphr4W9Kk824ervR7j3GNm/Jh70zy90rPBX0dZ27APw4g0KLSkD3bVyU/kLLwdT6w+i9Gx2VefcXvISNLQDFcRAEzixlKf/dKkevwpw9SOeP128LEbXkmdBfKajRP8cXSYGfI6/JadY95TSUfRTCLjjKk7tmRqe/0A/7VHipZrlP3GzrWDUvfx5D2WqGyBX1luNJlGplb3Ra1qoe4TQwTkDU7VlzUG2RMgztmAVU/Y5kbI1ZLQca749hdvm+zz4jAO+ebvCKIolak4Y/F/P/7x43yuLrRb516mkdIfekT0aK/F/4FRz/RK8meS/pT+gDMS9YyZNaeWkjgQaxOyxo9POg1oexxTOI+MsBCMUpEDNc/3OBXEh77sqh4En40xPhn5lfJsmjxLJju4OwTMS6qLrjn7eY6MIykzNGIi4UM2wjbog557JL/cjhReHy/aDomRRuSZ2ETJFWMgJXRXQAo7thzyDCiqdaz5OoNRj88AgbfjYtuRLEE4vmVnWMEzcPSI2wCRU17PurLuZoXm9691R+PXtQHa5khy6k1L7zH9iWO+C6INO/ZlkOFtyRfWW3U50vn2mpPy7Sh/cD/Xps7ZG3U9kvsL7VS4lFtXHRo82+Eo2TzlpI/ER7ZXkZe1ydKMYuycXPathOZ8Mg0T5nI++wrtjjmRw7roW7qQyb5eiV8CcQIt7SmnkK4PIMDKCy0dfSaL38RIpTCNkvyKc400DgSFWBXEMaESXmCJjSOYeeMHKx+elbgOvwnTg16P+W6nnMeYO4/tBvhxuDuc8g1SoxaoQtuxwQ8mQJdyZqoNywkdG/tBPm3bGWua8uqMok51k+P87cSCUhwfsvJxTcPCT10hOnFbSB2Ro1WUAqGMi1POvCvxZXD8h3bFszOQ6mIynsQxqdpxjJQ9jqVVg73qZ/8Pg183BQ=='}, 'original_manifest.csv': {'sha256': 'b2b4d559fcee93f4f93f8b613d204fe78d6d7f506685910917616ab3f8e83746', 'zlib_base64': '...'}, 'prepare_data.py': {'sha256': '4ed7d4cc66148fc1c917c41797433d7b8cc8886d585b2a148071b9cf7c6e024e', 'zlib_base64': 'eNqtGmtz2zbyu34Fyk6uZCszelqyc7oZN3GbzLWpJ0l71/o0HIgEJV4okgVI2YrH//12F3yLUpzOafIgAexise9d0DCM34QM/D1LN4KpLEnCQHiMS3cT7ATjkcdcKXgKj0yJhEt89ISXwToXnj3m8ZQrkdq93tuYxTJYBxEPWbDla6EAj2Db2At8QGqzD3cxE/cIGaThnu1w49puejl3XZEA4sseUrQJVApIXUA5H83ORJTKfYO6IFUwM8lnAKUK4ojdBemGTpRupBA9niQy3sFObpwEuE8IZ/L2TIotDtvs+xjWwxovcwULPEBGW0qR8iACuJu3P7LVPkXQyCPCFN8K5oaCR2zLo8AXKoUDwkSO+0zPFaQGikVx2uQcXwNyu2cYRs+X8ZY5jp+lmRSOA+xLYpnCZgDEUziR6hVDcg1SUKJ4d9WueNxwtQmDVfEaxMXTf1UcFc9xiUltsjQIi7dUbBM/CEu8nwL9SpS5cRgKl+goSHsZZ1EqZB+O5PMsTL3ATfViUAiRBsCdfGXx3mf476c4ypEmPEVyi2U38NpnN3D+m1gF9/jaK4iJsm0CYgceJhr25s1PBdwb1LRe7+rdy9dvfrt23r++Gk3P2YIZ7kSszmfj2WCyuhh7nLuCCz4azPnYG/reaDDyLgYDdyT8uTcYjt2L1WzMBzM+hj+jwcrzjd7VT++ur1797rz86frqrXO4w4pP3KlwxWw+m13457PVaHDhz8cXrnBnU3824HwyH7iD2cqdCzH0hBgM5v5kejHhF+fnw7EHO7x8eX3z4fpVgfw9oH3oMfg1d7tkRmUJTmFlRl8vPUEmAOYK6dR1DyAfez9fvX3zw/X7D/UTjVYTbzq98F0hLsb+xId/5iukdTSY+HBQ79yb+dPB+fl8ejEcXAxn58NzvoJFYj6eTc6NniaiA/V4MDwHCYjVnPuwdOqeT1bD6YU7Hc4vppMLn3ueGM+mY84Hc280Hgk+EzNvMh34EzHxfA9RX71/Tyy6NaJYbpEBzIhVKmI/SPf4gg5K7SNPqG051nhf9m6u3rwjHMQ700glWOFzje/56HxsJ9EawVrjExq3+l1Q80kn0HzaBVPS+3w4bMHVp0rYZe/63zfXL1FJXv7y69sPlYpoKOOS3Y6Gwz4bDQd9NoeH2WwJKHfAHpgaz/psPOmz4Qj/4gR4sRRnptM+m8LMaICwS1CIHtgyuAXuoOWbaJ/WJe1E7jRORESDgEOuDAvt0dfz+ANfmcmocEI2onC8AIJAavoohQ0HRTAseyPu82FLb+jyKI5Ir+V65SC8GWzzjWEEjhtsbTeOwLenpvHux+8Nq9exod7A9HHF5QMA2neBl24e7+l5I4L1Jn28NGwRubEnTIt9h9jtNCa/blodlG2BvSY43N3iLXitnCSeAEWFF7av5DrbQri4wTdpgqK5MkjQUy4cx4tdx7FyKBsU3OH5ctM4O8tDA4g/3Sdiof2fFH9mgRTe4oPMxFHQOEuTLD3zAvlXoIt41YTN/Ti9mQ6pABBvo+idCIKdaRRuxykDHgQfw8r3kWuFjElsYgxuqIh3ejqIgGBHywiWlUqGy+ycE/WVHwOI7Qt24B/ttUjNOrIcyK/DQbBFgdV0kwdKsN94mIlrKWNpGle1uCzgiEJSulCmJGUiA7bcnGnGd5VmXpmQ2EZJTfOABcMs9tWCtXzjKSpfV9lPgYK5G+F+VNmWbQO15am7KXYlI21sZ5PJRuIuhCRmYYC8SfmDaL0wstQ/m3cYMfeAFQtMK+xXENLf0YDpWy8YnD70UMYhUGXqlTYNonooWCHju9Z8yY8QCMFp4gBkcqdOfX2fQLIBrIZ1lSAwwYPcrTjtau+gL0JnKG8NKUJIk3aCxozlJZPMB8FJ0Aqi6rFORw76FFJeFRGz4j+CllTE2yBFSoGKFe3o9NkKN6Uo81j4KZ1EQsxpkYUUHVJPmSKsyJEvCcvXlFz6wT0g2nSoBfgaMnBF2lomvGXIB7oDqZjYiSjHd7cRES3Os+AyOcYkfKUAF6NcKz1IasnUSqUndHisj0Ik4EdknJQcqHgLJ+XRvuD9La5d3pL2LFEOxThCF+MlbP7DPWgG0ZtFRMGoD1wJneqdHD8+aK8PLuqygetAzn7d0posA2GwMI7WYBVkbUJdsgek/rHPHpDaR6Put3ITKIRbd2aL7hyOiVCJUkkIlcgNwNHgpHHdil6pU7V9oXUkVCeFPQHPLnapjkDfC1mz8Mxcu6wvIvNWCyWRYhfEmQr3TqE8T8HbmYk2ELtYVdBZgc234OVMOLOCtSmcFVBQ/YUm44ZcKYpLesKtGVbOSouGXBzKM8e2RuG8wvlWhlV6i5wc0M/WihM+wzd+jQoB6gKR/gdyc3SgP/qh0Bzy2TqkQ8YkMYCC8dnbj/Bi6hdFER389z2Ixok/1gI8Fm7AMArbRRGHsPhsgqDAZSwMfq8S7hQxzAHbANyL7o3zgA5hw8HaHnDTFs+ZkZf6Rq7vnoDwgrUyJW2CxPb4AnxFAsFPQd2KI7fLF9gBEE4p2rxyNHPy5b5iJoWwvPK0/wiSHw5SBIxXn5rWjIv13r22cIPIj1G+n2x8orjUcgVl8uDHNhgKcrxjhVbNKA2iTBxM5mGoUbmahBApQxW1urZEMNwSXG0cZilmpECxYdsGUkyzIA/gGIyGeeatRyhwTXC8GrsdkBuFkvQTh4o72rsbo/scpxW26FcgZlDUxjEKfW1gEyGafirN5vm/rZE2vFxanSxAYDgrCfDJxL4DraP2yR9vbthWbFdCAqGAqos8Qn0Lk0ssI+AwvRYJoM4mLSKmdjneQ8Ig8VKQQ1Uur9Nfn9VwHxKWVSwv0dRoOeskpYP9h/xBtmgKvJgyCQpbzRz2oXLMj1p/sYPHjE4ZNBlq/KxPDzzP+QAmX50GhquXtkQ6gtXlEY06DHUHC1dhjMXhJxuDillJulPVWlUiwjYKPpQ+hhqdQyyfrI3UfTrDErLMx49rI7k3gtCZeRDb32Px+eYXTRB5t2DbvTkUwWEMB7W6Z31coIJPAk8C7ifFyKkToaXVZ/lAnhAtrSNO7sgpXwWQWipq/3mB75+yODKQyt/fAlX4umTfLdiwm/SDQFIYbHdn4Nj5jSvynLAh1IgryBOUeeKUGNciqI0Vl5LvTQ3lbngUiRDKQ8OyjoNWAc6GRBtINx9a2nqJatz/vDnVftjNiZQOwY4vObVbAZEPUk9NIHULqYTJMdkZWNYX4kZHHAAfNelHsQP6fwB29jcGT39no+nUsqzHo767Wad089rD4mRRZRPPEfLoyi/MfTpR3MkgFY7u6pBZHfigk4VKIyv4fFpMdxCHGkz1DZpixxzVOE/2MFf5fciXVSP4W8s4SzAzqrXoTUyCqiUHSXOTLI3htstCD/zzclkYw+Gc1S4DMaHZWaBsQyJhhyTozewdHh77cZ+p2epZS1WyQTkC54AywU/zlg7dADEFK/PODoUtcGV2jVXkmc0iz6WsvdXl0h7buIPU+UsaKoQcFbLeVPkXDWBbtGqfLPL2ynfs1qjzGz1fEZWsDrRa3Te6TXMk3HYLt40DwrH58O23EurmIwRcsqdpQs1jNFnZ0QA8xfO61nQjAgPrvHD4nO78KCIheVmh1dooFN5U1fmoSt0qfWpd+9UIhYp1y+W+bNGXDrgsvbLUBT4Wd2J2BEwvrsVsmLOgJIh9vD+ApKTp5A0VZ9LFziql6HWhVO3Q0yDoykoAfGkt7/Bwh9s1L5iOY2jJCiBbQmqBdkoYoDrHj29LN89O3i0E8Plo1s2U9kL0SVViauEtR243ThYFf2Y5hMY5aRNP2Q5SSw99pjsUtDrvPrQg8qjplL7L0f03AGh1Up4I6EA4FgBtXBXNv9LVscJCWRqz1rV6wdg+hfNIuEJBThSE+e04D6lDgm0kVrWRbKNFVNFuOiCq1X+izKizM9VCeNhi6uLU4aoWGnEPqY7jylgph/pIFRYovTFI1LpRmr7bZbe+YBqLe1IYraW4qClVSgiMDVxcVssSu/El0VpnKU7xhQRA/QD8Fm1WUHsT3W0p4lq7st78RH4oIVH2EIa2oL6Yc71g3f6cZaCfTNuiufqG7s2KO7PivuwbCEgwoT9+sA7Erg8JWbMHEXGNlH3AO0l4fi4wkOueI9Yt6FRhP7q5Ly7zcMtvLOxFKZdLDKOaaxSO4yyFhdskVkEKMzb7FeiX4R5XIUU6T6BudWGnB+RBPArwzNX5XWLfL5HQX6iwRFAXBVexO46GkpDxwYF14zsDSmSDg7DGjaXH4MQfEQWCoenwHQ9CvgqFrWu8M5RYfl1T1OJgZ7AigPFi0zppB/TntT9ac8RhgRMGYPyFohrvwFiDnf48BbYQCWgmA76FdN2gEro3kWJNLkB/ElPVGGzF3Y+Yc0Xei/yWhcU+HJqnqEb0qQ8KL79TVNjDxBiWOwD2Nmbc8wJ80V4kg//A0JIEJQTeguffEm2EbJysatOZ9caiwzPAZuO3KpBt6SQ+FfepiSO2l20TZebhFdMuhR/KcOUGwULbDHEyShcjvNk1/hM104d2q5OKiYMCMQ69MtDoW9460JE0pd1SQjnXEZV7YceuzHvqK6j7kJ/t9kgc7CgYdFqDPdJr2qG8PkMfBDLAaghUgKQYFJU7sOgFKGUcK/ykC1JZpg+IDWFIpmO5t40vTiMPLjybTCv6xlj8HdYGf631Uh1TKysomrZorOHXAnQ6oxP6oLqbz54RgkSGSY8mu7xwSeyS2DQ+dTLL5spBV3VvWk+pzJGnVHyeRirXUL+axrcG3aYk2C4mHluPbaXrOsBXn7kyKuT5eNn7K+zOycTWWtkSBdS1zib15L5MEF+zf0I1S663vPlteiPyvXcbQJ2/4L7w7m50XttCVzS00UuXCbyXSYSipAaYI2Q9tWkg0CB6huTT6RUaqypUyjiJK+9ynOhrNCCQm7D9AUXPmXminFCp9HHENJ79/mz7zPvw7PWzn5+9d575fxjkKXOPe3rf/4c/7iiWG5smEpuUlaLpKqvQskJ2L6r79CJLgOgokT7MEmo48V6x5d4V2HMScleXnf22KCtYTUvtpA9GtQwz9FS2LRdQqzjcgXFiQtiuB044hc8XaU8uWU7tciTtzrEcmT2N8jD7zrEdTpxG1FVfMeOwFPxcjHysaZ0Wpo8eJNw3vkNA2R9JAPQ3sbbcplJoJbF6PQBx6KrZcagP6DjYY3Kc/IKNvhKzev8Dt8UmjQ=='}, 'preview_training_inputs.py': {'sha256': '1a5c92fd646779f586515678cbaccfa1582d92db426b740ab4dedd07c6feb718', 'zlib_base64': 'eNqNWN1y27gVvtdToL4hmVC0nU3SXTfMjHeTzWQmTTx22l6oGg5EQhLWEMElINtKJjN9iD5hn6TfAUBSlJ1d60ISwYPz//MBR0dHb4QV7UbW0lhZpqwSN0LpZiNqO9W12rEbabZcMV5XrN5uRCtLJutma1nTiqWSq7XNJpOrtb41TNeC2ZaDV71iFx/fsUa0rFTcmJSVetMoiGKtMPKLqI4bXlWiYkspVJU69rTdCIHVCSRLPPHtihThVuo6Y2+0MKzWlgmnCAcz3TLdiJrdcCUrR3Ys7mBPDY3lhq+EySZHR0eTZas3rCiWW7ttRVHgXaNbC6lg57aZSbfUrhreGtE9r7lZK7noHn8zuvbcGm7pRcfqAo8dEdRrdoxD2cbTXrz/0NG9J606Qqvbst+14bZR2pKw4W+2NSKOzlerKLlPlzU7+keSGmW9KNPAcQWiUJulbjemk3tF65fO9xccDv98ef7x6tdPl38vLi4/ff70y6cPKVtspaqKQw6TyS8fzq+u3l6xnM2iGktcRSmLtLFCL6Xd0QN8xs2uroTZ9Guj5/lkMqnEklUSQbExeS85mzB8WoGg1J2jM7Pmz168jMmfniprBa+Kxc4KEydJthZ3gUfSszSN4ruiXS1iK2qjW5jCy2tRV0GEXDL/grg3gv0lZ/EPKXv5/MR9JZRHgaCyO0/gYpMtleb2h2dEQJnnF6VZIsetCNKSjCsVB1HOIi6NYOfGiJZy623b6jaO/lGLu0aUFkm/0ZVQoYycRqg7J9eJqaeePWX1ViDwxPNa7OD/aKeVLnwM5BeXuRFZF8xlOUi2CrFTO1SziZhQ0CRyaqOQQX6w2fHeCF6zzmJvU3w/P2ZQYT6LiDiaJ7OzlH1ExfrvueNjbPVoNqD9DheEEVyCb7Oy2cYJe+J4P3WaJhnaymYL95+m7FnKTpLMFVycdLEGhwwdDftesempmL4kv7pFfofF1+wUrGj9z0L2Rkw7dx1ETaLbba2RlWCX737G9nrVRSrkc91kpZJNDLnQMWWnXbpuOKnmRbtW08Lcru1k5+1qSz3vwr2JUUBlKxvSKS+KSpdFkeztzNBECx62xNF06trvtNXaogIppXIqpBRK/b6Vrajyz+1W/CEHmAUDp5Vs/5QDthnoHhi5H2Jlhkgggsj3uMlqvhEM7YQ1cKDbmDlVC1I1Q7K3EIjYYE+D+ircU0J1GJrPYaj+SbVxv7L68ePGDiSqSrQmGvTNvH0kINtckxjoDdONsytl4g6DsNDXe2Yu5QoD6k44W5XNzHZBbdfEz1OGLoLX1FXz+BTJePoy6bKAFrFj3HhjajeBoNRtRTxnPu/JOxKDUhlyUT/j4uCAvf5i9LYtiXdwb3zgT3ZMXJJspfQijp5kTY3xkcxO5j2HW2nXfhRlND9jzzGhSaJbuZKYn4M4+nSrGfphFeLbfdyghTY9TanrG9RRHKE0ooG2vk1ZjUxSYmmRWrrBHu+mbCX0Rth2FzteGa0N+5AT1Htjql7PI6FGB0862ECV3i2/Zn6NRLDXeXgiUfQw0tolREf6FMqxVwNP2oG1dVhLxs54uFm8rx0KYbdrrcTU4ZpgHmEd1pm45xBTckWOI6nHjMz7jvl8YWAim3pXZ7eyQvie+P3O6OzFCT6n1Occ7bqnXQuCaA8RP8amd0FpVq6pwxk4h1u9kSUhQkM1h42gZwux0/CbrK1YoaG1eltXktKuF9KXaE6tkRvetnwXewd5u5MwBXO8D3M3gV+evXgxeMzBw8LXMNXO12/9O6qfbg6igOLRIExHQzA6CKigScu9cvto4mFAFP/KMVUHjNFr3y2Mq8NnL9msVKk08FwvLu29ggZjtcppKKHV0t/DpPt+kN56dlhgjWyEQ85Vh5WB0kF9Iwb/7+fnA0lJHz/CN7wG8i/I5/HpX0+eoyLkmC7g88f6zXfYx7ltw821TxUAAxPHPVbrkmShtbq/ZYbKPeuq17eas77C52DoYncvQAhOvYvpB9XTWzUjjnMHGBCXR8fjfO/UEgqnYgsFKxkdesZlcT+rZ8EfpO1+qvRqjVsD6X2QXxw+fSihDvWniiFaKpcvgCoHeowKaJ4eqjkqqHnyqB75szfNIFuWS/QJA4QvofeOOhe0oP5RG5wX0FCgnWBjtLpnuTvckYf2mkmXUENR/aFFs9P5wNBKqzzD6FKUQt4gZnSEPQbyhCIVwTzqIv+i6vGn1rNuyD89iC7IPu0fhfePsXs2kPt/S1nsT9ipVyEZD38Ki7c1vDeHjiZgMgNu+G2eyY3BOTzwS75HZYQtHKt4GX0FTvjG/vef/7KvbukbdF9qoCECND99lwUHSIojvVzuWRPwTMYbAIoqrmRpYwfCCsJ+OQTBAjSAQvGFULkkRKngkhtR0DEvX0bOX8dOpeOvHpE43AilxuXiToh5OAUG6JIGVFQ41RVA3P4wTbtrB/965vHCfMw3hK+gflGgh+SzHqkg+wNu9Pu7bjTvxZatboqFvsvdcWasr6dwOVPgdAArA4xOR6nhOm3eddoxD9/zojAWESWumjUv0Fwq9PtV/sA5KxqTUIl2cBYQtvEpEH0W5bp28/zgVsdlRZ/Cod7oRuhvbq74wxC8ipIy+1lz+nwQY4kRAr6D8zDqS5vPTvxhCN/ZTz+G+rPAr8L6U9A+QEfxRZ0GhVOvgHo3Utw6SDtYw28EfmPPBwOikfnpjxCz5CWyUuk2j27XOGSEdCUQ7zsldvmlQ0DsWTlADCgrEbuqkz2U3+GbfXCMscorbrmbjCgFgxhvTR41qAhRRUO6Nkra3JtJBy6kdOGdXZAiSAiPNvYgkIIj6TwYfcBhpdSrljdriiC66FK2xvZXb+PDEMWNsrjEyJ8uOLQYeO1VWD+vYZW2Gu7LH7orGsiQCVsFQ0ZV6a5vCuQ83hRJRu51fSCODlFB1gB9JIP84MvCn3sP2PYc054OmXBAFGKXhqTNQ2caRJRrUV6bnCpEdYepMIkYXR/WlnVl6hNjBHXpfPAqB4pu7gacm97DBhHOy+AX+B4Pk23vbo6ja+CUSmq4ejLd2CMIsKD70PH8e0BKuCrq7qn8rYkZZqC/Xzp2FxRO9W39MCQJtRg/vgbpLjRCbFsoUFg4LqaVrNpuGhN3+Y8o1BU8mgPLP2XRv7sB2LSS7hveHzQdXx5nziVLRGbIYXHH6QbZhNAwi5NNO1wru+7qUpxKqO9XroSykcyQHpPJBPCpcFlZFO7mrCjobqYoIl/i/qJm8n9hrYNh'}, 'spine_transforms.py': {'sha256': '0db3795dcb4818126e2e80af91ba83e696d2a5ae84f50e22f40c0dcc6f4a3b30', 'zlib_base64': 'eNqNVltv2zYUfvevOMgebG+KkgZJsaXzQ9s1XYYuCVxvewgMlZYoiyhFaiTl1C3y33cOKcmS4wZTjDgRD79z+87l6Ojon0JLfnx38x6cYcrm2pQWbMEMz2C1BSYl5Lo2YCuhOKSSWStykTIntIJKVFziexuPRotCWCh1VksOZW0dGF4yoWDFrcg4uIKTBqGEWp/wDZN1gLCpEZWz8FBwBVKzDM9HacHTz5UWytlL+Eia54jyld+xDFCL5UYwif97EwXeto6tSK9Xf+KtBMVKHo+Ojo5GudElJEleu9rwJAFRVto4YEpp562waP789c3Hq9v5n8nd/HZx+/b2A8zg2wjwGa+5Lrkz2/EljO8MR/Wb4BBXThgOGL1X6C5ZiC6oNUfvebbmEVStNLMVTzEmpC6CFC9yc1yhO06D/bfGcMfjKGgjGNT08vy0eYFiFJXErFf4/v40Avosm1NBUJWW3hFv4fUHWAlKCzMtJpNVwZKCqQzfr0nqN2FTZjLwJ/AgXKFrB6nG0FjhUIg8cpRA6zSRYf7+DVDaMNktapvPpNBGfNXKMZnkUlRJhRpO44t9sZUR68IhXaw/f7F/niKEYdYNTpGVmS6T1GhCvWLS8uEJy3P09cAZq9clBvrwLY6KQij6h1stdaKwBohebUS/jUvO1CD0mCWX0ZsXEdBn+dj6ok1abITFm8/hxOc/XyBQfH7x0n+dvuxhnsZnZ7/Q67Oz8/B1QfiPo9EoMHtYEZdeM/I8vOlyifWnN+giVukXLi1oAykyYE2vmGJOl1jGcsBMLGPCukZKrbmBTzEBfkKOWE2dQD9Y+EtiiuTWidSOn7QD5IlIMQSe1Vwirsdz1Bos2yCLuiYDQlnHsQB0jpZuuCHO0T0q54znrJauqROg3Metl8FElMCKRtq4JJmgqjwCsnWGVTMNAaFH5IAlji2DlDGV8gkJRajbTSkePl6/zuB0d4Ue5KPl8DeR/Z0x2kx8TYautsJahlAjWNYiRGo87e6TLT5s2D7oa2du20Uac0XJ1jwhkZ7BDyJzRQQFp0JBhJ1Q3ycv5e0mJxrh/+PGNcFBJrAsiKG2c6l1KO57guzwXnQenUDJvkwGNu7EQ//LkmDczItiaZRCTTqECIyuVRYg4MegYjp9CtIF4DmURuggDHZ6NTQp2gOPYIcIx0PZKZycwNl3JBrHvUifjGiE7MjoE9dLrB9B1Jmb2eMzscupz0uT7hjbINXDZIwtt5ePQdwjkDzH305XbYo6fgUUz6z9wHY6wv+TvWRGwa4YOwkrKxoV8ZvrD9c3717Pd1ApUxtmESnIKv4QLO2HK9qxBkEnTd+c7oPEFXZ7PmmMQ8HOqafJDDf6ATe8Mk3Ae5FuxPPxsEv6yp9968x69F1gNzlnraMR5ELKGdqMP2g8NZ+ZnxHTMXZgUr2qhcwSvxElu61p0s6xCFYs/cxV9qQnYfv6GFYrW6FWJk/IBqxFt8USlCLdvoLPnFd+uSCQlVb8mBo0NVkYDJSmV7/pZupJOz6hGXwexeLylvKGYSuOlvJABtrDms0ChaD5cxRCSJubBf6FpU5uYSXRlBgWBS4zgOaDVrRedPtcf3XEo+2uV3fU783FtgR6l5BOi1HT3JrYhcatcGjWvYkTDSbs+LGX9v1m1+AEHumKm7DpIW/vl62u1oEdTE/wJ5RcxHO/LvzeLThXuN9Mni6L988tQ8tpNGjMvWcRv8WVw/whHHJxsluQZs+q6C1Sy+9CD5+WHM8DdxvYchqC9ANc4YjBvU9kYcb7WY6fXBjrLvvrANSWWz++xd7+4HSDZXiq14raqadmWCyFyrnhOJx7y0FWG7+5hIXCs/1Adg5UeGP2oFIw5YecPrDr9fk3m8GAeoA7FH9uwVuODhJooRdY4UhI7CWL+Ka5wieDy/dhNcQlcO81rYRtLprORpShLZ1Pdrqmo/8AvFNznQ=='}, 'test_evaluation.py': {'sha256': 'f6def3b89a9e6275d15cc56497a89a89791ba30e9286c904fbaeb2286827595d', 'zlib_base64': 'eNqlV0uP2zYQvu+vIDYH2QEr2F5v2t3AhyJIi16KHpJDIRgETdE2NxKpkpR3jaL/vR8pWZafm0UMCJbImW+G8+bt7e2nv74SsZbimyNmSZznXjmvBC9ILpdKK6+MdoTrnFRcWZkTpZfSSi0kGbit9msJapJzz4fp7e3tjSorYz15ckbv3mugeOn8zW5B12W1JdwRXd0srSmJE6rapkG4Iy3NQmlTRq6G4lshudVpKb1VYk/FCw5VcsaFqC0XW+aEsZISYdZSs2+8qvhuaTnevVkjGK9F89lpJTe8qLmXrDS5LFzQD5s3ouDOkc/NJozxBTq5we5Mafj8xJ0cPt4Q/GA04qT/Wg2cLJbtYviJksxw3pRby7eDLLsf0ykd0dGcZnd0+kDHdIzXERbHH+IqFrA/fpjPhx2It7VfU1LBD0oEZYCZzSmejmRpLFFwErFcr+Rg2lNht/10ebsTksoXL3U+yNScvAe9H4gyU/RpPhx+7MnvyJ5OyDrcYIh02z99lACcuHNwmI5kv3oEVDVUy7ooBoNC6haMToeUpKPJMXUWEeNZ98RDeiR5DtD0YXrTuTC4lhVqYbndMr6yUpZSe4Y0YAurpGXC6A1WQnIcezpGlQOiTBfcqTbMGrJ024o+0hMhJq3//E/Ni0FDniW7kE7g4PThLMOvRWncEVutn6VarT2SopcEAeQkJw5V2lv8u2WdpF8QcyEnf1hYyYU1bDkOMna5fAGU8o20fCVnDVPyViFmY0OBCJIOasWhOFrWhVcsFohZAp7kB+QGqIjUBFgQPX3f7Uquo1Lt3vAoUhvGXKJkKs29sS6GqnzhwjNkpbQoXyeB6tcAX5sij7Eq1sY4yUDIuo1LMYtt6BvZdlXzyDQdBJ2M6OTnPas2tuQFJbCANNV662OqNIBZUoXUCodJ5tnjZH45SRqYLHFSO/SojfKIvSxBZ4EDYADY7348fDN/z4QB4f4KQu8A0LvanGH/Zc8uVGOt6JHY2xQwxnf3dHw/2pPJl0oKpC6IuwbYUSHATehUCHAm1ACtcG3yWRIxk1cjTagsEYoV5hmq7eSk+PxezjWqSp81fF+xz8lh0cwmw54Oo7cwjycdd6vHee/+4f40Wp4TPgI3rKlKBGsyvyI79IldPGIOCk0gmBw89MNx4tU6Tkmh2CHfcrmSOsSfZAtjvPOWVydZdy539nNBmAiGtDcnhCFg1A4E8S3FQED/TboESx7T+/+QZVdOtDtNOzyFSD2qc1kCVVTOrKwKDH842ysOOgN5qfG8AfuLreVZbU+qY8i1ztiyrJQNMyvLMbxatahbh3Uywjia5hg7HUJDFKGchqVW1pDyAjHJNNez31Ao5bGbw+ir9Co6OeoSToKScexblYeZIAzPwb3NwAxjlBW3Kog7rJHt31XXdYgwgtCy5Lat6v7ZMIfdnFUXk+EEAuZBR8m5bo70mpMPOJft2B8dsu/3+6SCg1sShhpYu/bgbTG8nu0I5JLx/KlGucvS0ZimoymeO4Q6Pu/w+iE8PY8a3DwszNwYMRWm2g72u+/Ib8biihKyRIZ5daNM7YotZiBrUb7aNtSbZB3xhiwkebZGr9JDMdnjXZgQocmEYkyk8X/Ua0/t1eh7nB4Br/u8gciSvT1brXsWZVFP2PXuTUCRqw/TIr/ioB3QtRBMJ/fHadPVKGbrQjIVRhTMIoiOeMc8zp535BP84BGd8ItZ8IUq0JoxH6zURuJ6Ju2WmCqkPHKRVAaDDfnb1AjS2ehjD6WZZOB7SUKvgCYk3mTlfuKhuNESXjzzrftJyxUQISGXQrlwqemwDm4tqM1jOqGISdyA+reQabh8xON3dUAvzeWhakvPj/+/W4k0soPAnfUKPGx7bRBpyEv+wrbRGK8ldYOuQpCinqk8pK4wtQ5BgPsT7sNLwlALS8kYmc1IwljJlWYsaVzVXX3DKnLuf1vcXM4='}, 'train_models.py': {'sha256': '2386fd1fbf253e58d577d891dc34d4ef369fbb35aa51d72561fb33a7392a17e1', 'zlib_base64': '...'}, 'trainer_notes.txt': {'sha256': '051bea78f18b9d8cc517cb3eef1546d46d795394f203093ded43f7ab6277d5eb', 'zlib_base64': 'eNqtWl1z27gVfcevwEtn2xnqy46TTTR5UGw5q1lH8kjypu2LBiIhiRuSYAnStvLUH9Ff2F/Scy8Akkq3b53ZTWKJBC7uxznnXni7ni2Wi+VnuX5aLudr+e9//ksu51/l/dPDw+B+MX+4k5vHxXIuH9er7ep29SDE9pRaif+ULPSLjDOtCp0MElUrqV9LXaW5LmqZNFVaHGWuisbGVVrWstLPqU1NEcnC0E+xedbVWZiDxIK1qdJYZbKuVFrgxaHcnrRUTX0ylVRxrMvayhofZarWFstjO6trqYoES6lEV9KetK6FsjI2FVaveYkKi+D/m3fv26Uj+f5GPqssxRowh5eY3IxhfK2rAjakuTpqOxRkAcw6pkUwbITXRrz/wWTY0/r9a3ynE/kIP+7P+J73LCttdfWsE7dSbPIy07Wmk+v02T9+SHWWkDfDIlM6pdXSmqaKNT1j5UklUmV0zLPYa13Q0iW2SLCbNE2d8bnYPZU+0pngVPpJv6qcLbMlH8LqIwVnKMTsafvLaj24fZitFwgzorx9uvubvJtvFp+XnAVX46u3g8l4MH7L5if6WWem5NiGw7+we61Nj7QHhVpl2VmmhfyaFol5sXL+Wmam0tUQn9SntBBaxSdZNLnmcONha6oaL8cZ1vELR/BVUafHxjT4KM1wKlUgIm6/3JDvaiN6IWzweCZVWVbmFdGrNRae3PzJuUHVfnVyI7vP6gz5gcDIDf8LS4gXJA6M1qXGHzgjXuU8YFvwQQRLXsnj+oADsXcPaYVMCGnFqYDMFvGJrE0oa02VcGAMrE50RrFCGiC35NLQoRKTI7CZid0xkOo2Tr+l9QBFVRXClllaS7KssWTtMviNvVKonPKTUj8x8A5VFVJT7bPUnrD7PxqUG58tPlWmMJk5nodydqjphQZLY1c9iE15RtbAqyqLfAr5dO45+FApdhMyVby/Gb1990Z+lJM3w8mfppSDzr+wFYl8/W509eZnWFPltOL1G/z4Rhpba1OeqDgiMbkavb+WyGBlz/C2zf03rhSvRj+/l5dfuAryaICAwPM5JQHF1uW5L93Y4ImaI03u4GiT73phEwcuBY3TNXw4BzVtoSv8cbY4SJmpgp0PmNrzF25RJJktKWtQEGdBBYezVb5yJYL5jfIZ1d7UvD5e5aUYMi0VLvzeh0qCwgoosdFwvsEOtm6S8w7nR2ENf7ceo7DvM4JexHrUf2Dn4Y5OQs9e4taOH0VamVL7jShTsFgKV7Z+63tjJg9N3VT0eNUUAu4jw71nX07G9pzVvUfHK8xLC3NTpIP1K+0AV5VJGldpzpY2ZZE7ae7JBHVsmvgE/7WWdUn4AyCfVJXEqKru2R0h864kngFyZOddgL+P26rRHEq4jMkIsRM58pxoxPGRzsvU1RZeBwIAewEBzk2M+PDDUK7ag7/quGHgggmRoAAHMrBNVreuviCCqVScNtIZYQmxgFSUNomJG0oGnYjWRji1F1uCDIdIIALG3Jxhifx2SOuamS0tDkhFNr4SLcwOvFEOmQKOgjQBuDgd8XlvIyHu16u/z5dyc7uYL7egh1v5+DBbintQkgMy+0H+bfWwmkyyQZxZ+We4KVfV+S8Rf3z1lj+O5Frbpa5vxpGcHw5pnMJe/Dz4NB6Ke3aD1nARUcIHOY4m0VV0Hb0Z4rVwCHpgTJlFBZeCK9tCdicCMMyy7GrsrBrR44AVwLLMGyJpADOg0L/yh8n+W5tewrGCsweu0vG30qQoT+BbrkArTQ4gLycRHqiylFJClwaPIkx16jGd7YgEG0JQ1a1Ch/C0AzQF7rfmAAcOBJY4sIOiH9BM4NUTqQSwh2c6xz6XZ4kkSZ9MVd5PLq88ZsLGjs9dIfVKlzBL7ulPyuEza6wqJBXROadIX9yllCymOiNZvv6yeph7qbhdz5ab+9X6y0aQ4tA7kGNh6XR2CJphmQNaYuUC8Eocj4ZqeDlpskUxZ3auQ5A3tBiyKf2uHwHtDElAlBLecQC7T6GBEBfKi/Q7ve8VUKu6SGyhPoU95yh8lAUOrHDsUiWeo9++Gb/ifwAXyvAIB1SmKfg7Qu4asbX1eHhTQgZk7ELkFIOeZ/JRTPWLoMMwBAOpZ2r8EzvZGpwan7hC13h21hxd7SqURaFF6yXKGS8LNMjR6eCgL0yRnT9IRDP9DklCIiBLS1l+HN4w8FDq2VrsgVCnutDWjki5YJVa/o5I6urjcBJKwR96KD+ZGlKhgm01AkqgfwBkUnaTkwXH3Z5YatoSyQL5S94pICzOsjTQEJAU7NqsPClyR5LaGMiMF16g+JCygkJgSIkUx6kkoY/v1p8/EbI3XioHyQFB5mQv6Rv3VQHdiQ4gY4odCsIXF38C6ZrMcEoj/e7VPOsb2JCRz2if2oyHwwnY2eCcrgXhFQQR1IIQFaAkc7Qx4FWY8AlpsTeFHnjQiS93iKQBNuftfnA944DYN8lR1+SBw4FErDVtBqIoLaM5arwCUZDlpEZCaMu01JTASHWuzbSAeJDMZMSKdFyKfk91Zvo1jc2xUuUJBrYLOYIoKQtJkg3FovYGYAnXUkQgQbgoJ2ys8W6fwnlbOhGOz+BLMtYlq6dgL0FdebI0Sg9n0HyWDVwrc9SG6usc+QARo+Igg8soIS+wlRUKxQLluIfLobttxJCNaj0ABurrK0mpZip8zjIg9AFk4UX1OkQ7oGWghgTf8klGJ2VPo664AuFzXln17HPUK8gAbsR/i7/OGcxca7yZb7f4e8PJ90E+ZVgyO+No9uchJPDb60gCN8rXCOdAmU/G4+jqzZiTApZT2RCZ4aNI+NRBcTe1iWRWjT8Ox5Moqw78d27Igib/OHx//S560VTKkHmxOuPr8fgmelFV3pTXEYrK7rLq473KLCI6+/LoqLztnpeKhYbLh34IQ8FBscRME4Ih2NW4q2ip9lCaQyJtJ3Nla/boCFAscSpVHflsFZoQBj1gZR11SOaCpIvnFK0Hc0bndEpPKEsEravID61YaEty8uvutyte5gf1cPHEZCqc89n311dRU6LSEQPv/83nO7nUpNDNM/z9R25OvHPJqSklPinQJoOVSJgdCoW/n0TE++cBEKzsgjqO/sv3JEl6snXUEwE2tJvOO6+la3471YZSl/ePOIU/jQe87gHHw4hSWgMPmuLA3U1tWpHJ2dzr3HqNyFDMaeLS6ZBLeVJp4l6q+OKi90Mf26D9OztNlZNhQ/FZF4wDhkODGm+BlAo47SqdkTwK44yn9YPLjM0vs6ubt5IqFEjsUpIUM7e5RgYRT3yuiZ9+sDWcut1M9/wvHtfztnhvV8vtevWwEVKWZxBS4R7cOR1LqmTAs6sBd153s+1sFGZLg4GDqQHkDs3HNvjE+0UPCI+FV4/aXo5HHHnhlC8VwMyOPFgyegMpUZV5Q3265sZw+H8yjRoi4IMewHmQBoAOZ+SqkJ8fn6IA2SrLDiTk0Yo01NX2yJ9lDlvIPTahacLF5YCN+/AX06N4yv0WG8DturRhnqEvvlDUEKIHnn+ZDUjFUd9UnkVv7sU1xMme6xzKkscnpHYJzagkeik5AFyVF3UDGDSuU+XA+xEekybnU68cuZnraW6eunCTfJFiwnWsrEmaOIaegmNk8PGu9bHrz3GQPalFV4mmNrHJRj30cwTV0jQ+qUg/0VZkOj6AOdgkNGP43mQk3g3qzU3Khm7w6qN4QA+Ykuqtz65/VfLYAI0hzuCNI/axtHTmtTxi64WVSjOKNKpkuZKbxQM6PHk/e3j4NLv9Vdw+3c0uVPqhdU5HLavVF5JxCLVXeC47RrRH1jjQcY2mR6/2GO5BrO5HAVwjS7Vsh8FMEkgvqg04/tZP0iyDP0kxdFbclLQm9kYqwMOui+4TOhKg8LrCac7IG5gIhz+RU0sDN0zTVQXJQWmRp5bhTic9cAxoyDxA/nEqjWBnvt4sNtv58nYuZ8s7uZ5vtrP1diOerINlxEnvjfn2k+WuXecl5OtdRQyLpPkdRdh1VlMcxg/KiRRcz1mJMHoN2ekUDLUcI99lyF7P5eebHtP96LzXWYmOlLnrZc6E39tRdeP1AvlpwC/6tey3tCwpRvckR0GKCo1qK7Au5tm8clo8hykni2TS3uYFGdAkaTcB6wkCSlnqFazvqTs4iVp+Ju6N1q6x48ZRV4MjEZPCm4JRx/Xa7uLA1l076E/6Rbm+TD4yAI9KiEpCjhEQ0wslVwlToUjtENkgz8IyXvpf9D3Uw7sLEleWiBTyiKdJ+7R+wT9FGIf5ApZeBbvGgftrpBShAY3oOhAhOV+0J+DSOpAfL2ZM1K5Ti1ZVTQkDdwqtH/LMjsJ0iiLlZxAicG7H3k5z2k4YW5oyUysAdQshyrn4Cu6QhVOY5H4axuSaiqmburaTaPLI4uBblzrcY/RcuAeC9PDZqSI/j2/PEfq6IKB2IdcYf/2NQeH7GA8nbdOkajf07fds7fwRiUrQdjEtSMhUVDRx7v3D6iuX8+pp+/iEat52KE7NENO/2yhcEeHAdFsFv5tKAdV6HYPkjo7OSB1oYbphj7vWagcr8p56KjBUeqAvumslhqy2u0ZZaDJG8OuO9qb+fsgdtgePXGzM0EGPOalCKMZDD8gV8Awng1zcXSg1omP0kcWZX2+lKxJ5rziNeQT2yLq47t2kxJpY3t3ghdFXN5j6KXSlYarvW2w/Z392gyzXxTisdkwakBTraf1duzPRKJauo4Av0WXrUSTdWL9lEDcLEY7e2gkqgxsQEQ/WCRxMMwEkIeDlSGF2cO1v9JqKq7U1Rud7Gmj6Ia8fqTJ/+XEvCQS+S6PxrUe9Kc8h3DyZNWGss8zPPowL/KBrpWs3VAhTcAS1nUQHM4Zi1c3IAdtOmDi/0E0PJTs+QTX1bhTa+wJy4lA6xkLdtOP2NjNpzMjFXJnvunBOZ4HK4ztuLM88rgpv3s6+hHHtVCBelhRAdx9gvR98AQafdVzCpQmZS3RBtNkbPq7cGAHPxFmTdAJrx7qakGHaE+a7UE38zWgY2+ep6EvunTsTlPf0D8aX06PveX7ocPyofyr+hzicQogMwKTdtWNZj/6bi6d4ZBcAyL0o/gfcTXXozHftiRHdpnTfhtk4n6/XyMGsJHU+n/b4uFcpozItfUlNL8DXkwg6SOY7YmKEmAC/I63uLIPFaOX0sKpolLjX9QtxQhjzED6SxEZDAPiqwi0ZSixoWa+AM9q7P520pfaXa26kNpR3rJIEAvXt8lHthwaQagF0bBCANGab3/6yXNzOHuRvc7oCv51tF6slA/3D4ssCOH8LCUBB4uKs4pRbTsIe+q0HtMJYkzmYQ6Bpxn7z7v3o/Q3dXeI50ruAFI9eVhx4DO+VR+RTlifdbq0u16If+53oj1p24fj8B1kKyZebb649lZ0iHzmJ7mYiQVk6Ecx3ol05tq2fvyBZa9WyRdu/8G8llAQp3cBv5KaH7LGhmPk5Nv86QOjRk3Br1G+1VO1aTL8cc4NrTdv2jo86T1F/AwKTE2FtqSBQOpWCwvRbXN4VTGWuSjvqkI4vv0TLX7bt7ahT3Lpfvejix+EPUOzu8PutY3sLtaEJi639/WE7UHXvNxdr/sCYS2SToyCSnMHDrKWWq63XUgHiqbNmQxd8twqB4DqqgDxt58jCHch5azK1H4r/AKJqAF4='}, 'study_design.py': {'sha256': '218300b6a46138fd1cb0bd4785b57da3d065169de22d45f36e510f7c38df40f2', 'zlib_base64': 'eNqtWN1v2zgSf/dfwcs9yAYSp2mabZKFH4ptenvAYa/YZg8HBIFAS6OYF4n0kVQaX5H//YafEmWt7XQ3D63Mj/me38zw6OjoSyHWUBLa6pWQpBBSQqGZ4D8SDk8giYSvkmkgQrIHxmlNKin+B5xQqVlFC61wBw/V9JmsqFqBmk8mt6vu/InSbbkhSlMNDXBNmPLMTiSshdRQzskNctqQqtWtBCQmW05aBYroFUwUrKnEy7i+lqJsrXREGbHn5HaF5FZQr1HSUuANLjSpmCaNKKH2stGSsIY+ACmppvPJ0dHRBJVoyJrqVc2WuGnkIJ/x58R/G1VwK/z8jxJ8Mrn59+ebn25vPuYfb778/W+/5F9+/vD24geyINnZ27Pi6t275fL9u/J9dVUtq/OL8+X55SV9976srq6qyzdvzis4Ly7fv4W3V8uKwtuLi4uqgsvL5fKHKpt8vPn04bd/3HrSSNNIM83zitWQ57O5BCXqJ5jO5mgNNKP/j5ySzBo4L0GxBz43kmaTyaSEiqgVRfmmRs3Z9YTgnwS0MA/azf0By8qemhtj5cuNBjWdzeYreC7ZAyg9nXmSBeWCs4LWuaExfaJ1CztpG3nmZduslTt8TBQaNH+EjVrcSvvb+VdItZhmx9kxya6z2bElue8PuMKAyakqGFt8orWC2Rx4ga4fl74WqJ0zlNV38YvgQXzzO5jdfmPopE6Z2XMstSv5y4KMx8V1VEFSpoD8y6h/I6WQ0+w3LuGBKQ0ypt5JIXjFZIML1qEnTk5iAiBzrP3Kwobj3CijrBDOaxqe9dRqz/jDImt1dXKZzaLQPj4eQE8zVaygoTkmncJcyqwSZ7vlVe3aJWsqnaOURS5Tt36XhfTP7fHs/i5D8UCaFQwuna8lPDHRqnqTwzNtGIcyuzfYYBLYujKJAHRGoOxQIu+jwSvJm7h7JfWipqzJRZVz+Jq3XIsWFS/zyBRjmyG62NOJFrNdZv1nwFSHkZSXHQYOsU6RplUaNxrKOCkxeBgvtLe8zz2nxHESoD70+wTzBjQ1WLiVBIGAyxuMtGHGJOy+JdCTO67Ztb99TAbbRgvcHTVxmu+ZWtdMGzFXojQEvXPssnGH37kfXNsfA9d/QhgNmRpcs67PpajhABaDC/cv3kc+iCBf15RHq5tvY/pjTzcfOg3r2YeigLUmgtcbUzMJCluYWIkYE6u3IWfinZKlaDHe8Bx7Ars8N4XRQmFgGfEwLMzi/gCFwoH9ULQzxHoaelZSaFGIGs99e7wmT6RC2R+P8QOTwAqNvUmDxcqAz6NNO9yYZuFeCEoMxgJFQ/zKW11ks5cAWIOCFu5ZSLT0LWAOye3M6ltc4qj4ibVTZNDpYjiRhqmG6mLVYWfHrqgB7dlQzioThYGpkWkrFcbP3h8ioHNkbJskVBgrvAAbQtSGFIaOAQpClWFr+jc1JnC8mivAFLGSvjFh1jch9mByk9vGzJ3INqIWZ2f1SVGrbJfAn91d19RZmDRsEkCs2LNhbInU8ECLDemM5RZylw/Sw/SwmscYJ4uFJ3GXaW8ql5LbxvVWuNuKkHujYKCytXmdQMi2wr+OZW4In5Jh3yiDH/44Hg9K9Xbr9Z1w/GpI/q6eYUxc0UoMxUNYd/G5//RIU5oxjrfwsHZ47jgbo/6MOX7iIXZ7tOqGKezuTL/Q8mJF+QNyccgEz2s8i+5f/E7l3kJKjETKN9Oo0KNNsRQwA9UAmrODe1SfUqF6BPCCWECSrnDJuEH+MaRIQtANbgm++WyMg88uCb/0eYYh0CUHsQOewTFs2MLIGjIobZqCUXwRjh7NuzrdL8KFaNY1mCK9qyY7uDAZhh58RV13kiU8QhFOFtNzaTFODh40G4QbzkWjmNd3TgeVu7zzkydaks6Qbqqn3ktmftX9WpSlLUK+rzsIrrO1ex/e32XhySLCMC5hEm4UU6fhI4+2sJP0sEqkTjCVIpFhqzC4GLgbQ15bIYaYt68wxImhZ1PsMEz5XoKv0mhxLRIj92zbC/xOvj8GOUn0/Amw88jFV95XMEhwjW2KbZ48sEYURV7Io2YF0yOT0850j7nZ68prUTxOzT8+PyX8t8WigD42i7v68F8BTyARjOcoHKJk8bgWiCvKvn3VtfgK5Y+ElegaphmoU/dwZt7INq51j614lCKgQFwI/bGrUykCxEMHZX9HYzFQ9XoQMq/Lzte1YVOfYZ2C/Q4qVueBm2IqD+f5VJF9wLa73Tu8TJoy7+o4CRKf9ALZiOKj0eYHbNy0MsrWjCzjc0CKMt57LvtgY/UZ0d5sHYwu5lYsl6dBgOHMYstljOIpdjdClqrH5a/kw1KJusUKVQiuUUms0C5DBMdUkGAHjy5DeuTmQ7i62x4AZTr9GTztaGUv9qy0B51oMdh6Ugf73WX+EPrbUEoU61uzf/Cgxx1zCQOik+zUvG8HUAsBk82GiOS6ltw8Z+ZSCN1vGczvQS9ioeh3wem7e5GD092IFFDKfHfPpTtK4GJXCexVpD2VPZKwxkowAWoFHcntp4vBA0cPMWezMUm+uWx1c6yarzd+isCwaDnvutn78SdsHJowC3KkwRWGZ5NQiKt+kNhLq//y36Ozo8m+f+mDRxJAKTqM95Yjnei4xfpWi09e0LMa+j7ySN7EVCHZWqdONInMaQPx2Wikq0hQ0cCKjUJyai/O5tjamZECsQJp+VrTP2AfVyzxPShZZZ/cOOF85PJatU1Ex2vyrUf4JW0/XL/1fy2pLZ4='}}
DESIGN_ASSET = {'sha256': '121c944bb74d7f9fbf353b388a47df99f8003fe3c872e29bfae2555ffe88bb6f', 'zlib_base64': 'eNqtV8uS5LgNvM9XKPo808P3Y/dkh332JyhAEuyWt0qqkFQ93bMx/+6k6r3uiyN8qZJEEgCTiQT455eue1ryK++pf+N5Gabx6bdOfm2f847moQ6ZVnztC62MoScllPsmxTfhnrZZy3Sc8zbyz/fDbsjD2tFxfZ3m7mF9N4zd+srdsh7LR3eY+UDzaSBPY/O8vfx+WYvxaV6/duMEc21x5ZnHzF2dp323THX9QTN367DnZaX9YXk+R4MQVgTzJ17wumdYKy24PY1H2vXjcc/zkPuZxhfuF95x3nb3Y1hfh7Fnyq894l6WzRwsYGKZ9sNPblYq7RY+DyDslce1H8bCB8bP2Pyu8/EyoQ7vXPrEdZq5r8O8rP060zAO48vjxJlpt5sAU5tOC34R0H4qvOun43o4rstfXCO84WXcN+/36A/LCtsd5cwH2AL+TDBE41AB0nNe3n4Hnt3IP7oNpm5YuhceeW6eny8bflzVL6+krGsOtJCOiTkFqtobm51J0sZsZYjWxEqlsPZWE4lQlFZMnn0xVlTDptRydTAdx21LpzPClw0WfLA+fr18e6MdvkR7/bAimkZNK7Yvv/D7azvyaR5ehhFnu1HrdvZLng4bMn/fjmBj32Vud9jROAKjur3x+8rz9gC3x42Jl2gvQ33z34O3b8N0XHYfPb/Tfhj/ixYYL42o/fpx8n4idA+6gAP7B9uHedjT/HE66zb3Y9pNUu6+5d2Nf3xmPsi6OROfBnZstME2z7wp/Ma76bBR5MrANvlug+e4rzjW43rEPGTePJVjPs/5K5h/Gz+QnPNx7KgigEdUb9a7qWIIBFtoz0hxQLD+L5DepcfNaD9Puy2I+xhbstzbOM0GGo9OkdTDvp9qD/r3IOB0hOiV/hoLlg3lc2B2/EL5A8C8IMNmegTmktJ9Y9RdtgThq6tUORjjtKs5xhITGc4BKVOzoZBDITImJpMCJ9LFGZa5+iyEMjeOTOuUp92daZVKtqbaqJQyTiXllMslYK0l04xzduSRmoKVk8lZbxIbUWuuNdqkLqaviZPBmZv9x7Q8EWp5PrTMeqopa6NsLTEmq3K1rISMQlSlkys+eiFt9tJgOISM9BeZsCJTkqokyU/XdL46v9en5iKpZIq1sWbmqKup+AnJSV2UMJV9KHBUrXAu2ChFlN5JRwmTOECW3M3Fqchwq1x0jt9wgSTl7Jw0oWaZsTwbicCN1sWnkHMIwRUbgBNhjvAyxXYmjnEod/Fv4PDco0Dx8ry+r6faKCCLimKNqhaE7k2VkQNAsYwDq5Y8xaC10r6SqT5Vl5LVvgSJ0MTNPA9Q5j7TvuXjjj7O8XsSKRZOSlnWLodklAPNMsCt4ECMWgbvtfZBl6SdsjlwNoa0JymUyE8PcnqnBpfzdYWrTiFSZKe8LYg8W2eiJ3wL0UcJogqOKrgccR6JWjDBBUyOttxt4JyHj/xBMdCerfHJGwl4FNZUFSVqCbZEMckiKNfoWMKBS6wrtoN/7IxBjE8AOltm8L2KlCsXFoVMZq1NyIREMcYUUMmAWY2ewIxstd6nLKt3pZAPOd0sLwccbCvV4wLh3F9CFwDUI1ysDzJI5VhxEFQbIkGzi67ALiHfQU5WqhqRRQHVKuBPWjzw8m2ACl2lYxhbkT/7kWRzY48zznswJjgrrWshorBXkjaoEnHeOHmAKAjZwqUIMLRy8jI8ncrjpVWY16FSXj9Jb2j48v1Bvp7/vWzS9v8Sr3v5ugSy9OjgfvLNV85a4JSEDxXIEqgaGKaqFBl5jxYDuFPMxWYW2gZuDyyITdYyS3GX7wQ5+ViG5ftdxagz80++OjMGMMoMh0FmcrYw40kzNXaRsTEKWWRQhB1aYAvFDMZHwx6DoepPnF0eoKL7w47XmzOIH5uKrFSBbPLQFFdiSNaYwLUkhbznrCvwtYmFiAqSlrKzbKVGkn22swMN81ba9lC2AX6WrS0QV58WFiT6KBbBJAcRzQpsgkaW7MEU9ALSFuObU+MgJUoXwZlTjZkkxOiTBPt+0yKc5huPhG7kRpUioTcRzMfhoyuE3GlNVqbgBPSNwOQSRYk+WfaxIAmbTicOFvXI6PiZw5PktfvI9fpw9aeKcEg+CgGbM8oHKqgBuVRdQrENRKQielFBxUhhQjTQPtQo5bxAd28e0+Na/u8og1b8j7tqW1nFAOprFZCULlKRhJLnk05Jx+QMZJY09gkBwDYZJdI4kSHpUN4S9NO1j8jTPJ+vG9d26h+85Hk4rMPb+Wr0rXBr7jvcXKgVrm4adx/P3b8uLdbMy3G3Ll+7TVe7H0Dttb1uc28XgwW3ptItecBzA7FrlDmup96s3ZyOY35td6Cy3QkuMoBW7u6udeC5KWBrk6f51M7d9vD89OXXl/8A2ehnFw=='}

def install_active_assets(project, assets):
    code = project / 'code_design_corrected'
    code.mkdir(parents=True, exist_ok=True)
    for name, asset in assets.items():
        raw = zlib.decompress(base64.b64decode(asset['zlib_base64']))
        assert hashlib.sha256(raw).hexdigest() == asset['sha256']
        target = code / name
        if target.exists() and sha(target) != asset['sha256']:
            raise RuntimeError(f'Existing code differs: {target}; choose a new project')
        if not target.exists():
            target.write_bytes(raw)
    raw = zlib.decompress(base64.b64decode(DESIGN_ASSET['zlib_base64']))
    assert hashlib.sha256(raw).hexdigest() == DESIGN_ASSET['sha256']
    target = project / 'study_design.json'
    if target.exists() and sha(target) != DESIGN_ASSET['sha256']:
        raise RuntimeError('Existing study-design metadata differ; choose a new project')
    if not target.exists():
        target.write_bytes(raw)
    return code

CODE = install_active_assets(PROJECT, ASSETS)
print('Corrected active sources verified:', len(ASSETS), 'files')


## 4. Zweryfikowany ZIP i przygotowanie zbioru

Wskaż `baza danych(2).zip`. W kolejnych sesjach zeszyt użyje zachowanej kopii. Oczyszczony ZIP zachowuje wszystkie 824 pliki. Przy historycznym ZIP 827 wpisów pomijane są tylko `train/normal/264.png`, `train/normal/85.png`, `train/osteofity/115.png`.

Kontrola integralności sprawdza wszystkie pliki i duplikaty. Nie wykonuje predykcji testowych. Trening i techniczny test GPU otrzymują wyłącznie dane rozwojowe.


In [ ]:
ACCEPTED_ARCHIVE_SHA = {
    'c4eb673704b93daaceaea208a3d1fd202d900c2ef8d013c9b73a07a37a320bdf': 'oryginalne 827 obrazów',
    'ba4c5ece78779f67b209f839cec75f70aa480c07bc8ee1dee008f4594a96613d': 'oczyszczone 824 obrazy',
}
SOURCE = PROJECT / 'source' / 'baza_danych_verified.zip'
SOURCE.parent.mkdir(exist_ok=True)
legacy = SOURCE.parent/'baza_danych_original.zip'
if not SOURCE.exists() and legacy.exists():
    assert sha(legacy) in ACCEPTED_ARCHIVE_SHA, 'Nieznana starsza kopia ZIP'
    shutil.copy2(legacy, SOURCE)
if not SOURCE.exists():
    from google.colab import files
    uploaded = files.upload()
    matches = [name for name, blob in uploaded.items() if hashlib.sha256(blob).hexdigest() in ACCEPTED_ARCHIVE_SHA]
    if len(matches) != 1:
        raise ValueError('Prześlij jeden zweryfikowany ZIP bazy. Nie znaleziono jednoznacznego zgodnego pliku.')
    temporary = SOURCE.with_suffix('.tmp')
    temporary.write_bytes(uploaded[matches[0]])
    assert sha(temporary) in ACCEPTED_ARCHIVE_SHA
    temporary.replace(SOURCE)
    del uploaded
source_hash = sha(SOURCE)
assert source_hash in ACCEPTED_ARCHIVE_SHA, 'Zmienione archiwum źródłowe'
print('Wejście:', ACCEPTED_ARCHIVE_SHA[source_hash])
run_logged([sys.executable, str(CODE/'prepare_data.py'), '--archive', str(SOURCE),
    '--output-dir', str(DATA)], 'prepare_data')
AUDIT = PROJECT / 'data_audit'
AUDIT.mkdir(exist_ok=True)
for name in ['clean_manifest.csv', 'data_audit.json']:
    target = AUDIT / name
    if not target.exists():
        shutil.copy2(DATA/name, target)
    elif name.endswith('.csv'):
        assert sha(target) == sha(DATA/name), 'Niezgodny zapisany manifest'
for proof in (DATA/'verification_invocations').glob('*.json'):
    target = AUDIT/'verification_invocations'/proof.name
    target.parent.mkdir(exist_ok=True)
    shutil.copy2(proof, target)
print('Gotowe: train 579, val 95, test 150. Źródłowy ZIP zachowany.')


## 5. Rzeczywiste wejście modeli — pełne pole PNG

Podgląd pokazuje po jednym obrazie **wyłącznie z train**: źródłowe RGB, wejście po proporcjonalnym skalowaniu i dodaniu marginesów oraz jedną deterministyczną augmentację. Kontrola numeryczna sprawdza geometrię bez przycinania, kształt i typ tensora, zakres wartości, czarne marginesy i zgodność transformacji obu rodzin modeli przed normalizacją.

W treningu dopuszczamy odbicie poziome oraz niewielką losową zmianę jasności i kontrastu (0,1). Nie stosujemy RandomResizedCrop, CenterCrop, RandAugment, losowego wymazywania ani przekształceń afinicznych. Są to augmentacje podczas uczenia; nie zmieniają źródłowych PNG. Kanał alpha jest odrzucany bez kompozycji, zachowując zapisane RGB. Normalizacja wejścia pozostaje właściwa dla danej rodziny wag ImageNet.


In [ ]:
run_logged([sys.executable, str(CODE/'preview_training_inputs.py'),
    '--train-root', str(DATA/'dataset'/'train'), '--output-dir', str(PREFLIGHT)], 'input_preview')
from IPython.display import display, Image as NotebookImage
display(NotebookImage(filename=str(PREFLIGHT/'training_input_preview.png'), width=1000))
assert json.loads((PREFLIGHT/'training_input_preview.json').read_text())['status'] == 'passed'


## 6. Zamrożenie protokołu i środowiska

Przed długim treningiem zapisujemy kompletny plan 20 przebiegów, hashe kodu przygotowania/treningu/analiz/Eigen-CAM, konfigurację analiz i środowisko. To plan nowego eksperymentu podczas rewizji, nie historyczna prerejestracja.

YOLO: 240 epok, batch=100. Torchvision: do 100 epok, batch=32, zakończenie po 20 epokach bez poprawy. Wspólne są geometria, podstawowe augmentacje i zasada wyboru checkpointu. Optymalizacja, normalizacja oraz budżet epok nadal różnią się, więc porównujemy całe procedury uczenia. Walidacja wyboru checkpointu i inferencja używają FP32.


In [ ]:
CODE = install_active_assets(PROJECT, ASSETS)

EVALUATION_CONFIG = {'bootstrap_replicates':2000, 'batch_size':32,
                     'reference_model':'yolo11l-cls', 'reference_seed':0}
PINS = {'torch':'2.11.0+cu128','torchvision':'0.26.0+cu128','ultralytics':'8.4.163',
        'numpy':'2.2.6','scipy':'1.15.3','pandas':'2.2.3','scikit-learn':'1.8.0',
        'matplotlib':'3.10.7','pillow':'11.3.0','opencv-python':'5.0.0.93'}
env_command = "import json,platform,importlib.metadata as m,torch; print(json.dumps({'python':platform.python_version(),'packages':{p:m.version(p) for p in " + repr(list(PINS)) + "},'torch_cuda':torch.version.cuda,'cudnn':torch.backends.cudnn.version(),'gpu':torch.cuda.get_device_name(0),'gpu_total_bytes':torch.cuda.get_device_properties(0).total_memory,'gpu_capability':torch.cuda.get_device_capability(0)}))"
runtime = json.loads(subprocess.check_output([sys.executable,'-c',env_command], text=True))
for package, version in PINS.items():
    actual = runtime['packages'][package]
    assert actual == version, f'Niezgodna wersja {package}: {actual}; wymagana {version}. Uruchom poprawioną komórkę instalacyjną.'
assert runtime['torch_cuda'] == '12.8', f"Niezgodny build CUDA PyTorch: {runtime['torch_cuda']}; wymagany 12.8. Uruchom poprawioną komórkę instalacyjną."
protocol_artifacts = {'code_sha256':{name:asset['sha256'] for name,asset in ASSETS.items()},
    'clean_manifest_sha256':sha(DATA/'clean_manifest.csv'),
    'evaluation_config':EVALUATION_CONFIG, 'runtime':runtime,
    'study_design_sha256':sha(PROJECT/'study_design.json'),
    'study_design_scope':'future_reproduction',
    'experiment':'Reproduction on the same already evaluated cohort; no new untouched external validation'}
protocol_path = PROJECT/'protocol_artifacts_frozen.json'
if protocol_path.exists():
    previous = json.loads(protocol_path.read_text())
    if previous['artifacts'] != protocol_artifacts:
        raise RuntimeError('Kod, konfiguracja albo środowisko różni się od zamrożonego eksperymentu. Nie zmieniono zapisanych danych.')
else:
    payload = {'created_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(), 'artifacts':protocol_artifacts}
    temporary = protocol_path.with_suffix('.tmp')
    temporary.write_text(json.dumps(payload,indent=2))
    temporary.replace(protocol_path)
if (RUNS/'external_evaluation_started.json').exists():
    print('Ewaluacja już rozpoczęta: plan i trening są zamknięte. Przejdź do ewaluacji.')
else:
    run_logged([sys.executable, str(CODE/'train_models.py'), '--data-root', str(DATA/'dataset'),
        '--output-dir', str(RUNS), '--validate-only'], 'freeze_training_plan')
print('Zamrożone środowisko:', runtime['gpu'], runtime['packages'])


## 7. Techniczny test GPU przed właściwym treningiem

Wszystkie cztery architektury wykonują na danych rozwojowych pełny batch treningowy, propagację gradientu i dwie rzeczywiste, odrzucane aktualizacje optymalizatora oraz inferencję o kształcie walidacyjnym w FP32. Sprawdzane i zapisywane są pamięć GPU oraz kompletność testu. Techniczne modele są odrzucane; właściwe 20 przebiegów zacznie od świeżych wag ImageNet. Zbiór zewnętrzny nie jest odczytywany.

Jeśli test nie przejdzie, nie uruchamiamy długiego treningu i nie zmniejszamy po cichu batcha. GPU może zabraknąć pamięci mimo poprawnego kodu — to musi sprawdzić właściwa sesja Colab.

AMP może pominąć pierwsze aktualizacje podczas zmniejszania skali. Test liczy wykonane kroki za pomocą hooka optymalizatora i zapisuje każdą próbę oraz zmianę skali. Limit wynosi 32 próby; utrzymująca się niestabilność, nieprawidłowa wartość funkcji straty (NaN/Inf) i brak pamięci nadal zatrzymują test.


In [ ]:
if not (RUNS/'external_evaluation_started.json').exists():
    run_logged([sys.executable, '-u', str(CODE/'train_models.py'),
        '--data-root', str(DATA/'dataset'), '--output-dir', str(RUNS),
        '--hardware-preflight-only'], 'gpu_hardware_preflight')
else:
    print('Ewaluacja rozpoczęta; technicznego treningu nie powtarzamy.')


## 8. Właściwy trening — cztery modele × pięć seedów

`START_TRAINING = True` uruchamia uzgodniony eksperyment po zaliczeniu poprzednich kontroli. Można wskazać podzbiór modeli lub seedów na daną sesję, ale plan pozostaje kompletny, a ewaluacja wymaga wszystkich 20 przebiegów.

**Po rozłączeniu:** wykonaj komórki od początku, z tym samym projektem i środowiskiem. Zakończone przebiegi są sprawdzane i pomijane. Torchvision wznawia stan z końca epoki. Jeśli dopasowanie modelu już się skończyło, a przerwano tylko końcowy eksport/walidację, kod odtwarza ten etap bez ponownego treningu. Przerwany w trakcie dopasowania przebieg YOLO zostaje zachowany w `interrupted_attempts/`, a następna próba zaczyna od ImageNet z tym samym seedem. Nie deklarujemy dokładnego wznowienia YOLO, którego nie potwierdziliśmy. Nie kasuj zapisanych prób.

Trening może wymagać wielu sesji. Żaden wynik zewnętrzny nie służy do wybierania próby.


In [ ]:
START_TRAINING = True
MODELS_THIS_SESSION = ['yolo11l-cls','yolo26l-cls','resnet50','efficientnet_b0']
SEEDS_THIS_SESSION = [0,1,2,3,4]
if (RUNS/'external_evaluation_started.json').exists():
    print('Trening zamknięty: ewaluacja zewnętrzna została już rozpoczęta.')
elif not START_TRAINING:
    print('Trening wyłączony ustawieniem START_TRAINING=False.')
else:
    run_logged([sys.executable,'-u',str(CODE/'train_models.py'),
        '--data-root',str(DATA/'dataset'),'--output-dir',str(RUNS),
        '--models',*MODELS_THIS_SESSION,'--seeds',*map(str,SEEDS_THIS_SESSION),
        '--restart-incomplete-yolo'], 'study_training')


## 9. Ewaluacja wszystkich zamrożonych modeli

Ta komórka odmawia pracy, dopóki wszystkie 20 przebiegów nie zostało ukończonych i zweryfikowanych. Przed inferencją testową utrwala wybrane wagi i blokuje dalszy trening w tym projekcie. Próg binarny wyznacza osobno z walidacji i zapisuje przed testem.

Powstają: identyfikatory i prawdopodobieństwa dla każdego obrazu, nieważona kappa, accuracy, macro-F1, balanced accuracy, metryki klasowe, AUC, kalibracja i obie jawnie opisane konwencje Brier. Seed 0: pełne bootstrapowe CI (2000 powtórzeń); pozostałe seedy: estymaty i dokładne CI proporcji. Trzy porównania YOLO11–pozostałe modele: sparowane różnice i dokładny McNemar z korektą Holma. Pięć seedów ocenia stabilność uczenia na tych samych obrazach, nie zwiększa liczby pacjentów.

CI na poziomie obrazów zakładają niezależnych pacjentów. Poprawność danych i ich przyporządkowania potwierdził autor; nie jest to niezależna weryfikacja powiązań dokumentacji pacjentów. Grupa morfologiczna syndesmofitów/parasyndesmofitów nie jest automatycznie rozpoznaniem axSpA.

Po ukończeniu ewaluacji ponowne uruchomienie tej komórki tylko sprawdza jej zgodność z zamrożonymi danymi, wagami i zapisanymi predykcjami. Zachowuje gotowe wyniki. Niepełny lub niespójny zapis ukończonej ewaluacji zatrzymuje dalszą pracę.


In [ ]:
# Preserve completed evaluation on reconnect. No completed results are overwritten.
if (ANALYSIS/'analysis_complete.json').exists():
    guard = "import sys,json; sys.path.insert(0,sys.argv[1]); from study_design import interpret_evaluation; from evaluate_models import validate_training_freeze; plan,freeze=validate_training_freeze(sys.argv[2]); print(json.dumps(interpret_evaluation(__import__('pathlib').Path(sys.argv[2])/'training_plan.json', __import__('pathlib').Path(sys.argv[3])/'analysis_complete.json'),indent=2))"
    run_logged([sys.executable,'-c',guard,str(CODE),str(RUNS),str(ANALYSIS)], 'verify_completed_reproduction')
else:
    run_logged([sys.executable,'-u',str(CODE/'evaluate_models.py'),
        '--data-root',str(DATA/'dataset'),'--training-root',str(RUNS),
        '--output-dir',str(ANALYSIS),'--batch-size',str(EVALUATION_CONFIG['batch_size']),
        '--bootstrap-replicates',str(EVALUATION_CONFIG['bootstrap_replicates'])], 'study_evaluation')


## 10. Nowa, odtwarzalna Figure 5 — Eigen-CAM

Kod używa głównego YOLO11l-cls seed 0. Przechwytuje rzeczywisty tensor wejściowy i aktywacje w tym samym wywołaniu co predykcja. Mapa jest nakładana na rzeczywisty tensor po proporcjonalnym skalowaniu całego PNG i dodaniu czarnych marginesów. Zapisywane są parametry tego odwzorowania; nie rozciągamy mapy na inną geometrię. Dobór przykładów jest deterministyczny: do czterech poprawnych (po jednym z każdej klasy) i do czterech błędnych; przy braku błędów liczba paneli jest mniejsza.

Eigen-CAM jest wizualizacją aktywacji, niezależną od wybranej klasy. Nie dowodzi przyczynowej interpretacji modelu ani klinicznej trafności jego rozumowania. Powstają ryciny 600 dpi/PDF, surowe dane map, hashe źródeł, warstwa i reguły wyboru przykładów.

**Poprawka odtwarzania partii:** moduł Eigen-CAM odtwarza te same partie po 32 obrazy (ostatnia 22), ich skład i kolejność co zakończona ewaluacja. Przed tworzeniem paneli sprawdza prawdopodobieństwa i decyzje dla wszystkich 150 obrazów. Tolerancja porównania pozostaje taka sama. Pierwotny zamrożony kod i wyniki są zachowane; poprawka i jej audyt trafiają do osobnego katalogu. Wcześniejsza przerwana próba ryciny jest zachowywana.


In [ ]:
# Batch-faithful qualitative replay, using this reproduction's frozen active sources.
# For the original study use provenance/original_code instead (README command).
if (CAM/'artifact_hashes.json').exists():
    for relative, expected in json.loads((CAM/'artifact_hashes.json').read_text()).items():
        target = CAM/relative
        if not target.is_file() or sha(target) != expected:
            raise RuntimeError(f'Existing CAM artifact changed: {relative}')
    print('Completed CAM outputs verified; no inference or overwrite.')
else:
    run_logged([sys.executable,'-u',str(CODE/'eigen_cam_replay.py'),
        '--dataset-root',str(DATA/'dataset'),'--training-root',str(RUNS),
        '--evaluation-root',str(ANALYSIS),'--output-root',str(CAM),
        '--frozen-code-root',str(CODE),'--batch-size',str(EVALUATION_CONFIG['batch_size'])], 'eigen_cam_replay')


## 11. Paczka wyników do dalszej korekty artykułu

Lekka paczka zawiera kod, manifesty, konfiguracje, techniczne kontrole i logi, wszystkie końcowe predykcje, metryki, ryciny i dokumentację treningu. Pełne wagi oraz źródłowy ZIP pozostają w katalogu projektu na Drive; nie kasuj go. Paczka nie zawiera źródłowej bazy, ale ryciny Eigen-CAM zawierają wybrane obrazy medyczne — traktuj ją jak pozostałe niepubliczne materiały badania.

Po ukończeniu prześlij powstały `axspa_clean_results_for_review.zip`. Wtedy aktualizujemy artykuł, tabele, suplementy, flowchart, Figure 5, checklisty i publiczny kod na podstawie rzeczywistych wyników, również jeżeli są niższe. Dopiero potem tworzymy nowe wydanie i DOI. Stare wyniki nie stają się dowodem nowego treningu.


In [ ]:
complete = ANALYSIS/'analysis_complete.json'
assert complete.exists() and json.loads(complete.read_text())['n_evaluated_runs'] == 20, 'Najpierw dokończ ewaluację'
assert (CAM/'artifact_hashes.json').exists(), 'Najpierw dokończ generowanie Eigen-CAM'
REPORT_ZIP = PROJECT/'axspa_clean_results_for_review.zip'
items = [PROJECT/'protocol_artifacts_frozen.json', PROJECT/'study_design.json']
for folder in [CODE, PROJECT/'data_audit', PREFLIGHT, PROJECT/'execution_logs', PROJECT/'technical_repairs', ANALYSIS, CAM]:
    if folder.exists():
        items += [p for p in folder.rglob('*') if p.is_file() and '__pycache__' not in p.parts]
for p in RUNS.rglob('*'):
    if p.is_file() and p.suffix.lower() in {'.json','.csv','.txt','.yaml','.yml','.py'} and '__pycache__' not in p.parts:
        items.append(p)
with zipfile.ZipFile(REPORT_ZIP.with_suffix('.tmp'), 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(set(items)):
        z.write(p, p.relative_to(PROJECT))
    index = [{'path':str(p.relative_to(PROJECT)), 'sha256':sha(p), 'bytes':p.stat().st_size} for p in sorted(set(items))]
    z.writestr('review_package_index.json', json.dumps(index, indent=2))
REPORT_ZIP.with_suffix('.tmp').replace(REPORT_ZIP)
print('Gotowy plik:', REPORT_ZIP, 'Rozmiar MB:', round(REPORT_ZIP.stat().st_size/1e6, 2))
from google.colab import files
files.download(str(REPORT_ZIP))


## Source and result provenance

The executable installation, metadata, completed-result guard and CAM replay cells were updated for the 2026-10-06 author clarification. Embedded payloads match the active repository scripts exactly. The model fitting, preprocessing, checkpoint selection, thresholds, seeds, evaluation formulas and bootstrap settings are unchanged. The previous notebook and immutable executed sources remain in `provenance/`; see `notebook_code_integrity.json` and `study_design_correction.json` for file/cell hashes and the exact scope. Running this notebook later creates a reproduction and cannot recreate an untouched external cohort.
